In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display

In [2]:
## ══════════════════════════════════════════════════════════════════════════════
## CARGA · Los dos entregables del modelo de proyección
## ══════════════════════════════════════════════════════════════════════════════
## Salen de `MODELO_ML_PROYECCION_ASIGNATURAS.ipynb` (fases 7b a 7e).
## El diccionario campo por campo está en `DICCIONARIO_ENTREGABLES_PROYECCION.md`.
##
## ⚠️ AMBOS usan separador `;` y decimal `,` — sin esos dos parámetros, pandas lee
##    una sola columna de texto y los decimales se pierden.
## ──────────────────────────────────────────────────────────────────────────────
import os, time

CARPETA = 'data/raw'
LEER = dict(sep=';', decimal=',', encoding='utf-8-sig', low_memory=False)

# proy : 7 períodos futuros — LO QUE SE VA A PROGRAMAR
# serie: 30 períodos (2022-2027), real vs proyectado — LA EVIDENCIA para auditar
proy  = pd.read_csv(f'{CARPETA}/PROYECCION_INSCRITOS_2026_2027.csv', **LEER)
serie = pd.read_csv(f'{CARPETA}/SERIE_COMPLETA_REAL_VS_PROYECTADO.csv', **LEER)

# vistas de trabajo
hist = serie[serie['TIPO'] == 'REAL'].copy()          # 2022-2026, con INSCRITOS_REAL
fut  = serie[serie['TIPO'] == 'PROYECTADO'].copy()    # los 7 futuros, en formato serie

print("═" * 78)
print("ENTREGABLES DEL MODELO DE PROYECCIÓN")
print("═" * 78)
for _n, _d, _f in [('proy ', proy,  'PROYECCION_INSCRITOS_2026_2027.csv'),
                   ('serie', serie, 'SERIE_COMPLETA_REAL_VS_PROYECTADO.csv')]:
    _t = time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(f'{CARPETA}/{_f}')))
    print(f"  {_n}  {len(_d):>7,} filas × {_d.shape[1]:>2} col   generado {_t}   {_f}")

# ── Grano y llaves ───────────────────────────────────────────────────────────
DIMS   = ['PERIODO', 'MATERIA', 'CURSO', 'COMPONENTE', 'CAMPUS']
SERIE4 = ['MATERIA', 'CURSO', 'COMPONENTE', 'CAMPUS']
print(f"\n  Grano de ambos: {' × '.join(DIMS)}")
print(f"  ¿llave única?   proy {not proy.duplicated(DIMS).any()} · "
      f"serie {not serie.duplicated(DIMS).any()}")

# ── Las trampas del entregable, en orden de gravedad ─────────────────────────
print("\n" + "─" * 78)
print("⚠️ TRES COSAS QUE HAY QUE SABER ANTES DE USARLOS")
print("─" * 78)
print(f"""  1. NO sumar `P50` para totales: supone que TODAS las asignaturas se dictan.
        Σ P50           {proy['P50'].sum():>11,}
        Σ P50_ESPERADO  {proy['P50_ESPERADO'].sum():>11,.0f}   ← la correcta ({proy['P50_ESPERADO'].sum()/proy['P50'].sum()-1:+.1%})

  2. `MATERIA`+`CURSO` es la REGLA de malla, NO el código de Banner. Para
     programar va `CODIGO_PROGRAMABLE`: {(proy['CODIGO_PROGRAMABLE'] != proy['ASIGNATURA_REGLA']).mean():.1%} de las filas llevan uno distinto.

  3. Las secciones se dimensionan sobre `P50` (no sobre el esperado) y vienen en
     dos versiones porque ninguna acierta sola:
        N_SECCIONES_TEORICAS_PROYECTADAS {proy['N_SECCIONES_TEORICAS_PROYECTADAS'].sum():>8,.0f}  la norma    → abre de MENOS
        N_SECCIONES_PROMEDIO             {proy['N_SECCIONES_PROMEDIO'].sum():>8,.0f}  la práctica → abre de MÁS

     ⚠️ `N_SECCIONES_TEORICAS` (sin sufijo) usa los inscritos REALES, así que aquí
        está VACÍA: estos períodos no han ocurrido. Vive en `hist`, y es la que
        permite separar el error del MODELO del error del ESTÁNDAR.""")

# ── Los tres criterios de conversión, medidos sobre el histórico ─────────────
_h = hist[hist['N_SECCIONES_REALES'].notna() & hist['N_SECCIONES_TEORICAS'].notna()]
if len(_h):
    _r = _h['N_SECCIONES_REALES'].sum()
    print("\n" + "─" * 78)
    print("SECCIONES · qué tan bien convierte cada criterio (sobre períodos cerrados)")
    print("─" * 78)
    print(f"  {'criterio':<34} {'Σ sec':>9} {'vs real':>9} {'exacta':>8}")
    print(f"  {'REAL — lo que se abrió':<34} {_r:>9,.0f}")
    for _c, _e in [('N_SECCIONES_TEORICAS',             'norma · con inscritos REALES'),
                   ('N_SECCIONES_TEORICAS_PROYECTADAS', 'norma · con el P50 del modelo'),
                   ('N_SECCIONES_PROMEDIO',             'ocupación observada')]:
        print(f"  {_e:<34} {_h[_c].sum():>9,.0f} {_h[_c].sum()/_r-1:>+8.1%} "
              f"{(_h[_c] == _h['N_SECCIONES_REALES']).mean():>7.1%}")
    _ex_r = (_h['N_SECCIONES_TEORICAS'] == _h['N_SECCIONES_REALES']).mean()
    _ex_p = (_h['N_SECCIONES_TEORICAS_PROYECTADAS'] == _h['N_SECCIONES_REALES']).mean()
    print(f"\n  📌 Las dos primeras son la MISMA fórmula con distinta demanda: los "
          f"{_ex_r - _ex_p:.1%} que")
    print(f"     las separan son el COSTO DEL MODELO. El {1 - _ex_r:.0%} que falta hasta "
          f"el 100% es")
    print("     error del ESTÁNDAR, y ningún modelo de demanda lo arregla.")

# ── Vistas disponibles ───────────────────────────────────────────────────────
print("\n" + "─" * 78)
print("VISTAS DISPONIBLES")
print("─" * 78)
print(f"  proy   {len(proy):>7,}  los 7 períodos a programar")
print(f"  hist   {len(hist):>7,}  períodos cerrados, con real Y proyectado → auditar")
print(f"  fut    {len(fut):>7,}  los mismos 7 de `proy`, en formato de serie")
print(f"\n  períodos en proy : {sorted(proy['PERIODO'].unique())}")
print(f"  años en hist     : {sorted(hist['ANIO'].unique())}")

print("\n" + "─" * 78)
print("CALIDAD DECLARADA DEL MODELO (para no sobre-interpretar)")
print("─" * 78)
_w = hist['ERROR_ABS'].sum() / hist['INSCRITOS_REAL'].sum()
print(f"  WAPE sobre el histórico reconstruido : {_w:.3f}   (~{_w:.0%} de error por fila)")
print(f"  Cobertura de la banda P10–P90        : {hist['DENTRO_BANDA'].mean():.1%}")
print(f"  Confiabilidad del entregable (% P50) :")
_c = proy.groupby('confiabilidad', observed=True)['P50'].sum()
for _k in ['alta', 'media', 'baja']:
    if _k in _c.index:
        print(f"     {_k:<6} {_c[_k]:>9,}  ({_c[_k]/proy['P50'].sum():>5.1%})")
print("\n  📌 El error es alto por fila y bajo en el agregado: sirve para dimensionar")
print("     un total institucional, y exige criterio para una asignatura concreta.")


══════════════════════════════════════════════════════════════════════════════
ENTREGABLES DEL MODELO DE PROYECCIÓN
══════════════════════════════════════════════════════════════════════════════
  proy    22,142 filas × 36 col   generado 2026-08-31 15:12   PROYECCION_INSCRITOS_2026_2027.csv
  serie   78,125 filas × 35 col   generado 2026-08-31 15:12   SERIE_COMPLETA_REAL_VS_PROYECTADO.csv

  Grano de ambos: PERIODO × MATERIA × CURSO × COMPONENTE × CAMPUS
  ¿llave única?   proy True · serie True

──────────────────────────────────────────────────────────────────────────────
⚠️ TRES COSAS QUE HAY QUE SABER ANTES DE USARLOS
──────────────────────────────────────────────────────────────────────────────
  1. NO sumar `P50` para totales: supone que TODAS las asignaturas se dictan.
        Σ P50             1,126,278
        Σ P50_ESPERADO    1,026,533   ← la correcta (-8.9%)

  2. `MATERIA`+`CURSO` es la REGLA de malla, NO el código de Banner. Para
     programar va `CODIGO_PROGRAMABLE`: 20.

In [3]:
## ══════════════════════════════════════════════════════════════════════════════
## RESUMEN EJECUTIVO — lo que hay que saber antes de usar la proyección
## ══════════════════════════════════════════════════════════════════════════════
## Audiencia: 👔  (el resto del notebook es 🔧 🧭 — para quien audite el modelo)
##
## Responde tres preguntas y nada más: ¿sirve? · ¿cuánto confío? · ¿qué NO hacer?
## Sin siglas, sin métricas técnicas, sin tablas de diagnóstico. Todo el detalle
## que respalda estos números vive en las celdas siguientes.
##
## 🔧 POR QUÉ SE CALCULA AQUÍ Y NO SE COPIA: si las cifras se escribieran a mano
##    quedarían viejas en la próxima corrida y nadie lo notaría. Se recalculan
##    desde los mismos datos, con las mismas fórmulas que las celdas de abajo, y
##    al final hay un guard que compara contra ellas cuando ya se ejecutaron.
## ──────────────────────────────────────────────────────────────────────────────
def _mil(x, d=0):
    """Formato es-CL: punto para miles, coma decimal.

    Es para un lector de negocio, no para un notebook: `1,026,533` se lee mal
    en Chile. El intercambio pasa por un separador provisional porque hacer los
    dos reemplazos directos se pisaría a sí mismo — la coma recién puesta
    volvería a convertirse en punto.
    """
    return f'{x:,.{d}f}'.replace(',', '@').replace('.', ',').replace('@', '.')


_R = {}
_rb = hist[hist['INSCRITOS_REAL'].notna() & hist['PROY_ESTANDAR'].notna()].copy()
_rb = _rb.merge(pd.read_parquet(f'{CARPETA}/DEMANDA_PROCESADA.parquet',
                                columns=DIMS + ['ins_lag_1y']), on=DIMS, how='left')
_rb = _rb[_rb['ins_lag_1y'].notna()]          # sólo donde la vara existe


def _w(y, f):
    y, f = np.asarray(y, float), np.asarray(f, float)
    return np.abs(f - y).sum() / np.abs(y).sum()


# ventaja sobre "repetir el año pasado", al grano en que el modelo predice
_R['ventaja'] = 1 - _w(_rb['INSCRITOS_REAL'], _rb['PROY_ESTANDAR']) / \
                    _w(_rb['INSCRITOS_REAL'], _rb['ins_lag_1y'])

# desvío típico según el tamaño de lo que se mire
_NIV = [('un período completo',            ['PERIODO']),
        ('una sede en ese período',        ['PERIODO', 'CAMPUS']),
        ('un componente de esa sede',      ['PERIODO', 'CAMPUS', 'COMPONENTE']),
        ('una asignatura concreta',        DIMS)]
_R['niveles'] = []
for _n, _by in _NIV:
    _g = _rb.groupby(_by, observed=True)[['INSCRITOS_REAL', 'PROY_ESTANDAR']].sum()
    _g = _g[_g['INSCRITOS_REAL'] > 0]
    _e = (_g['PROY_ESTANDAR'] / _g['INSCRITOS_REAL'] - 1).abs()
    _R['niveles'].append((_n, _g['INSCRITOS_REAL'].median(), _e.median()))

_R['total'] = proy['P50_ESPERADO'].sum()
_R['periodos'] = proy['PERIODO'].nunique()
_g2 = proy.groupby('horizonte')['P50_ESPERADO'].sum()
_R['pct_prelim'] = _g2.get(2, 0) / _g2.sum()
_R['per_prelim'] = sorted(proy.loc[proy['horizonte'] == 2, 'PERIODO'].unique())
_R['exceso_p50'] = proy['P50'].sum() / proy['P50_ESPERADO'].sum() - 1
_hs = hist[hist['N_SECCIONES_REALES'] > 0]
_R['sesgo_sec'] = (_hs['N_SECCIONES_TEORICAS'].sum() /
                   _hs['N_SECCIONES_REALES'].sum() - 1)

# ── LA SALIDA · sin tablas, sin siglas ──────────────────────────────────────
print("═" * 76)
print("PROYECCIÓN DE DEMANDA DE ASIGNATURAS 2026–2027")
print("═" * 76)
print(f"""
  {_mil(_R['total'])} inscritos proyectados, repartidos en {_R['periodos']} períodos académicos.


  ¿SIRVE EL MODELO?

     Sí. Se equivoca {_R['ventaja']:.0%} menos que la alternativa sin modelo,
     que es suponer que se repite lo del año anterior.


  ¿CUÁNTO PUEDO CONFIAR EN EL NÚMERO?

     Depende del tamaño de lo que se mire. Mientras más fino
     el corte, menos confiable la cifra:
""")
for _n, _tam, _err in _R['niveles']:
    print(f"       {_n:<28} ~{_mil(_tam):>9} alumnos    se desvía {_err:>5.0%}")
print(f"""
     Un total institucional se puede presupuestar. Una asignatura
     concreta se revisa con la sede antes de programarla.


  TRES COSAS QUE HAY QUE SABER ANTES DE USARLA

     1. Dos de los {_R['periodos']} períodos son PRELIMINARES.
        {' y '.join(str(p) for p in _R['per_prelim'])} — el {_R['pct_prelim']:.0%} del volumen — no se apoyan en datos
        reales sino en una proyección previa. Sirven para dimensionar
        un orden de magnitud, no para comprometer secciones ni contratos.
        Se deben volver a proyectar cuando cierre 2026.

     2. Hay DOS columnas de proyección y no son intercambiables.
        Para sumar totales va `P50_ESPERADO`. Para dimensionar una
        asignatura va `P50`. Usar la primera para dimensionar abre
        secciones de menos; usar la segunda para sumar infla el total
        en un {_R['exceso_p50']:.0%}.

     3. El número de secciones sale {abs(_R['sesgo_sec']):.0%} por debajo de lo real,
        y eso NO viene del modelo: viene del estándar. Aplicando la
        norma a los inscritos VERDADEROS pasa lo mismo. La conversación
        pendiente es el estándar de 80 alumnos por sección en teoría.
""")
print("═" * 76)
print("  El detalle que respalda cada cifra está en las celdas siguientes.")
print("═" * 76)

# ── GUARD · que este resumen no se despegue de las celdas de abajo ──────────
# Sólo corre si esas celdas ya se ejecutaron en esta sesión.
_chk = []
if 'skill_por_nivel' in globals():
    _f = skill_por_nivel[skill_por_nivel['nivel'].str.contains('Materia×Curso')]
    if len(_f):
        _chk.append(('ventaja sobre el naive', _R['ventaja'], _f.iloc[0]['skill']))
if 'metricas_horizonte' in globals() and 2 in metricas_horizonte.index:
    _chk.append(('% preliminar', _R['pct_prelim'],
                 metricas_horizonte.loc[2, '%_volumen']))
if _chk:
    _mal = [(n, a, b) for n, a, b in _chk if abs(a - b) > 0.005]
    if _mal:
        print("\n  ⚠️ ESTE RESUMEN NO CUADRA CON LAS CELDAS DE ABAJO:")
        for _n, _a, _b in _mal:
            print(f"     {_n}: aquí {_a:.3f} · abajo {_b:.3f}")
        print("     Alguna fórmula se editó en un solo lado.")
    else:
        print(f"\n  ✓ verificado contra {len(_chk)} cifras de las celdas siguientes")

resumen_ejecutivo = _R


════════════════════════════════════════════════════════════════════════════
PROYECCIÓN DE DEMANDA DE ASIGNATURAS 2026–2027
════════════════════════════════════════════════════════════════════════════

  1.026.533 inscritos proyectados, repartidos en 7 períodos académicos.


  ¿SIRVE EL MODELO?

     Sí. Se equivoca 11% menos que la alternativa sin modelo,
     que es suponer que se repite lo del año anterior.


  ¿CUÁNTO PUEDO CONFIAR EN EL NÚMERO?

     Depende del tamaño de lo que se mire. Mientras más fino
     el corte, menos confiable la cifra:

       un período completo          ~   44.250 alumnos    se desvía    6%
       una sede en ese período      ~    9.888 alumnos    se desvía    8%
       un componente de esa sede    ~    1.420 alumnos    se desvía   12%
       una asignatura concreta      ~       23 alumnos    se desvía   29%

     Un total institucional se puede presupuestar. Una asignatura
     concreta se revisa con la sede antes de programarla.


  TRES COSAS QUE HA

In [4]:
## ══════════════════════════════════════════════════════════════════════════════
## MÉTRICAS DEL MODELO — según la Guía Maestra de DS (T08 §5 · T17 §9-10)
## ══════════════════════════════════════════════════════════════════════════════
## Audiencia: 🔧 🧭 👔
##
## QUÉ MIDE: la calidad del pronóstico contra los períodos ya cerrados, con las
##   métricas que el T17 §10 prescribe para forecasting — y en el orden que manda:
##   PRIMERO la vara (seasonal naive), después el modelo.
##
## ⚠️ REGLA DE ORO Nº1 (T17 §10): "si no le ganas al seasonal naive con holgura,
##    no tienes modelo — tienes decoración". Por eso el baseline abre el informe
##    y toda métrica del modelo se reporta CONTRA él, nunca sola.
##
## ⚠️ Estas cifras NO son las del backtesting de la FASE 4g del notebook de ML.
##    Aquí el modelo se reentrena una vez por AÑO (así viaja en `serie_completa`),
##    no por período: el corte es más grueso y el error sale algo mayor.
## ──────────────────────────────────────────────────────────────────────────────
_m = hist[hist['INSCRITOS_REAL'].notna() & hist['PROY_ESTANDAR'].notna()].copy()
_y, _f = _m['INSCRITOS_REAL'].values, _m['PROY_ESTANDAR'].values
_e = _f - _y

# el naive estacional: los inscritos del mismo TT del año anterior
_nv = pd.read_parquet(f'{CARPETA}/DEMANDA_PROCESADA.parquet',
                      columns=DIMS + ['ins_lag_1y'])
_m = _m.merge(_nv, on=DIMS, how='left')
_cn = _m['ins_lag_1y'].notna()

print("═" * 80)
print("MÉTRICAS DEL MODELO DE PROYECCIÓN")
print("═" * 80)
print(f"  {len(_m):,} filas · {_m['PERIODO'].nunique()} períodos cerrados "
      f"({_m['PERIODO'].min()}–{_m['PERIODO'].max()}) · {_y.sum():,.0f} inscritos reales")

# ── 1) LA VARA — primero el baseline, como manda la regla de oro ─────────────
print("\n" + "─" * 80)
print("1 · LA VARA — seasonal naive (T17 §10, regla de oro nº1)")
print("─" * 80)
_yn, _fn = _m.loc[_cn, 'INSCRITOS_REAL'].values, _m.loc[_cn, 'ins_lag_1y'].values
_mae_naive = np.abs(_fn - _yn).mean()
_wape_naive = np.abs(_fn - _yn).sum() / _yn.sum()
print(f"  cobertura del naive          : {_cn.mean():>7.1%}  ({_cn.sum():,} filas con lag)")
print(f"  MAE  del naive               : {_mae_naive:>7.2f} inscritos")
print(f"  WAPE del naive               : {_wape_naive:>7.3f}   ← la vara a batir")

# ── 2) MÉTRICAS PUNTUALES — la tabla del T17 §10 ─────────────────────────────
print("\n" + "─" * 80)
print("2 · MÉTRICAS PUNTUALES  (🔧 T08 §5 · T17 §10)")
print("─" * 80)
_mae  = np.abs(_e).mean()
_rmse = np.sqrt((_e ** 2).mean())
_wape = np.abs(_e).sum() / _y.sum()
_smape = (2 * np.abs(_e) / (np.abs(_y) + np.abs(_f))).mean()   # simétrico, T08
_sesgo = _e.sum() / _y.sum()
# MASE con el MAE del naive como denominador (Hyndman & Koehler, 2006)
_ec = (_m.loc[_cn, 'PROY_ESTANDAR'] - _m.loc[_cn, 'INSCRITOS_REAL']).abs().mean()
_mase = _ec / _mae_naive
print(f"  {'métrica':<10} {'valor':>9}   {'lectura'}")
print(f"  {'MAE':<10} {_mae:>9.2f}   inscritos de error por combinación")
print(f"  {'RMSE':<10} {_rmse:>9.2f}   penaliza los errores grandes")
print(f"  {'WAPE':<10} {_wape:>9.3f}   Σ|error| / Σ real — robusto con ceros")
print(f"  {'sMAPE':<10} {_smape:>9.3f}   simétrico; inestable cerca de cero")
print(f"  {'MASE':<10} {_mase:>9.3f}   " +
      ("✅ < 1: le gana al naive" if _mase < 1 else
       "⚠️ > 1: NO le gana al naive in-sample"))
print(f"  {'sesgo':<10} {_sesgo:>+9.3f}   " +
      ('sobre-proyecta' if _sesgo > 0 else 'sub-proyecta') +
      f" {abs(_sesgo):.1%} en el agregado")
print(f"\n  ⭐ VENTAJA SOBRE LA VARA: WAPE {_wape:.3f} vs {_wape_naive:.3f} del naive"
      f"  ->  {1 - _wape/_wape_naive:+.1%}")
print("     📌 Es la única comparación válida entre corridas: el WAPE suelto")
print("        cambia con el universo de series, la ventaja contra su propia vara no.")

# ── 3) PRONÓSTICO PROBABILÍSTICO — T17 §9 ───────────────────────────────────
print("\n" + "─" * 80)
print("3 · LA BANDA  (🔧 T17 §9 — regla de oro nº4: la decisión vive en un cuantil)")
print("─" * 80)


def _pinball(y, q, alpha):
    d = y - q
    return float(np.maximum(alpha * d, (alpha - 1) * d).mean())


_cob10 = (_m['INSCRITOS_REAL'] <= _m['PROY_BAJO']).mean()
_cob90 = (_m['INSCRITOS_REAL'] <= _m['PROY_ALTO']).mean()
_cobint = _m['DENTRO_BANDA'].mean()
_pb = np.mean([_pinball(_m['INSCRITOS_REAL'], _m[c], a)
               for c, a in [('PROY_BAJO', .10), ('PROY_ESTANDAR', .50), ('PROY_ALTO', .90)]])
print(f"  {'cuantil':<12} {'real':>8} {'teórico':>9}  {'desvío':>8}")
for _e2, _r2, _t2 in [('P10', _cob10, .10), ('P50', (_m['INSCRITOS_REAL'] <= _m['PROY_ESTANDAR']).mean(), .50),
                      ('P90', _cob90, .90), ('intervalo', _cobint, .80)]:
    _ok = '✅' if abs(_r2 - _t2) <= .05 else '⚠️'
    print(f"  {_e2:<12} {_r2:>7.1%} {_t2:>9.0%}  {_r2-_t2:>+8.1%}  {_ok}")
print(f"\n  pinball loss (media de los 3 cuantiles): {_pb:.3f}")
print(f"  ancho medio del intervalo P10–P90       : "
      f"{(_m['PROY_ALTO'] - _m['PROY_BAJO']).mean():.1f} inscritos")

# ── 4) DÓNDE FALLA — el error no es uniforme ────────────────────────────────
print("\n" + "─" * 80)
print("4 · DÓNDE SE CONCENTRA EL ERROR  (🧭 T08 §6: la métrica global esconde)")
print("─" * 80)
_m['_tramo'] = pd.cut(_m['INSCRITOS_REAL'], [0, 10, 30, 100, np.inf],
                      labels=['1-10', '11-30', '31-100', '>100'])
_g = _m.groupby('_tramo', observed=True).apply(
    lambda x: pd.Series({
        'filas': len(x),
        'inscritos': x['INSCRITOS_REAL'].sum(),
        'WAPE': (x['PROY_ESTANDAR'] - x['INSCRITOS_REAL']).abs().sum() / x['INSCRITOS_REAL'].sum(),
        'sesgo': (x['PROY_ESTANDAR'].sum() - x['INSCRITOS_REAL'].sum()) / x['INSCRITOS_REAL'].sum(),
        'en_banda': x['DENTRO_BANDA'].mean()}), include_groups=False)
_g['%_volumen'] = _g['inscritos'] / _g['inscritos'].sum()
print(f"  {'tamaño':<9} {'filas':>8} {'%vol':>7} {'WAPE':>8} {'sesgo':>8} {'en banda':>9}")
for _k, _v in _g.iterrows():
    print(f"  {_k:<9} {_v['filas']:>8,.0f} {_v['%_volumen']:>6.1%} {_v['WAPE']:>8.3f} "
          f"{_v['sesgo']:>+8.1%} {_v['en_banda']:>8.1%}")
print("\n  📌 El WAPE global lo domina el tramo de más volumen. Las series chicas")
print("     tienen error relativo enorme pero pesan poco — por eso se reporta")
print("     también el desglose, no solo el número único.")

# ── 5) LAS 5 REGLAS DE ORO — checklist del T17 §10 ──────────────────────────
print("\n" + "─" * 80)
print("5 · CHECKLIST DE LAS REGLAS DE ORO  (T17 §10)")
print("─" * 80)
for _n2, _regla, _ok, _ev in [
    (1, 'Le gana al seasonal naive con holgura', _wape < _wape_naive,
     f"ventaja {1 - _wape/_wape_naive:+.1%}"),
    (2, 'Validación temporal, jamás barajar', True,
     "walk-forward, 7 orígenes; aquí reentreno anual"),
    (3, 'Features conocidas ANTES del pronóstico', True,
     "CONTEXTO §10; el notebook aborta si entra una prohibida"),
    (4, 'Se pronostica el intervalo, no solo el punto', True,
     f"P10/P50/P90 + CQR; cobertura {_cobint:.1%}"),
    (5, 'Validado al horizonte REAL de la decisión', True,
     "h1 y h2; FACTOR_H2 mide la degradación al encadenar"),
    # ⚠️ El tomo tiene SEIS reglas, no cinco. La 6ª pide, además del protocolo
    #    común, medir la DEGRADACIÓN POR PASO DEL HORIZONTE (Tashman, 2000).
    #    Está medida (ML celda 22: h1→h2 = +16,5%), pero el ensanchamiento de
    #    banda que de ahí sale NO llega completo al entregable -> ver la celda
    #    «HORIZONTE», que lo cuantifica.
    (6, 'Mismo protocolo + degradación por horizonte', False,
     "backtest h1→h2 +16,5% ✓, pero la banda del h2 queda sub-ensanchada"),
]:
    print(f"  {'✅' if _ok else '⚠️'} {_n2}. {_regla:<44} {_ev}")

print("\n" + "═" * 80)
print("👔 EN UNA FRASE PARA EL NEGOCIO")
print("═" * 80)
print(f"""  El modelo se equivoca ~{_wape:.0%} por asignatura, pero solo {abs(_sesgo):.1%} en el
  agregado: sirve para dimensionar un total institucional y exige criterio para
  una asignatura concreta. Le gana al "igual que el año pasado" por {1 - _wape/_wape_naive:.0%},
  que es la única comparación que no depende del universo de series.""")

metricas_modelo = dict(mae=_mae, rmse=_rmse, wape=_wape, smape=_smape, mase=_mase,
                       sesgo=_sesgo, wape_naive=_wape_naive,
                       ventaja=1 - _wape / _wape_naive, pinball=_pb,
                       cob_p90=_cob90, cob_intervalo=_cobint)
print("\n  → disponible como `metricas_modelo`")


════════════════════════════════════════════════════════════════════════════════
MÉTRICAS DEL MODELO DE PROYECCIÓN
════════════════════════════════════════════════════════════════════════════════
  55,983 filas · 23 períodos cerrados (202205–202615) · 2,612,770 inscritos reales

────────────────────────────────────────────────────────────────────────────────
1 · LA VARA — seasonal naive (T17 §10, regla de oro nº1)
────────────────────────────────────────────────────────────────────────────────
  cobertura del naive          :  100.0%  (55,983 filas con lag)
  MAE  del naive               :   18.55 inscritos
  WAPE del naive               :   0.398   ← la vara a batir

────────────────────────────────────────────────────────────────────────────────
2 · MÉTRICAS PUNTUALES  (🔧 T08 §5 · T17 §10)
────────────────────────────────────────────────────────────────────────────────
  métrica        valor   lectura
  MAE            16.59   inscritos de error por combinación
  RMSE           42.73 

In [5]:
## ══════════════════════════════════════════════════════════════════════════════
## HORIZONTE — qué parte del entregable está ENCADENADA (T17 §10, regla de oro 6)
## ══════════════════════════════════════════════════════════════════════════════
## Audiencia: 🔧 🧭 👔
##
## ⚠️ «horizonte» AQUÍ NO SIGNIFICA «pasos adelante». El nombre engaña. En este
##    proyecto marca de dónde sale el lag que alimenta la predicción:
##
##      horizonte 1  el lag `ins_lag_1y` es un dato REAL — el año anterior ocurrió
##      horizonte 2  el lag es una PREDICCIÓN del propio modelo (encadenamiento)
##
##    Por eso 202720 es h2: se apoya en 202620, que a su vez es proyectado. Y 202710
##    es h1 pese a estar más lejos en el calendario: se apoya en 202610, que es real.
##    La distancia temporal no manda; manda si hay un eslabón predicho en la cadena.
##
## LA REGLA DE ORO 6 (T17 §10, Tashman 2000): «mismo protocolo para todos los
##   modelos comparados […] mide además cómo se DEGRADA el error por paso del
##   horizonte, no solo el promedio agregado».
##
##   Se cumple en la CELDA 22 del notebook de ML («BACKTESTING DE HORIZONTE 2»),
##   que reentrena 4 pares (A → B) y compara el mismo período con lag real contra
##   lag predicho. Esta celda NO lo recalcula —no reentrena nada— sino que trae el
##   resultado y responde la pregunta que falta: **¿cuánto del entregable cae en
##   el tramo degradado?**
##
## ⚠️ CONSTANTES SINCRONIZADAS A MANO desde la celda 22 del ML. Si se reentrena el
##    modelo hay que actualizarlas aquí. El guard de abajo verifica que el
##    ensanchamiento de la banda en el CSV siga siendo coherente con `FACTOR_H2`;
##    si salta, es que las constantes quedaron viejas.
## ──────────────────────────────────────────────────────────────────────────────
# ── Resultados del backtesting de horizonte (ML, celda 22) ──────────────────
BT_H2 = {
    'pares': [('202320→202420', 2955, .428, .480), ('202420→202520', 2612, .406, .476),
              ('202310→202410', 3123, .421, .474), ('202410→202510', 2700, .401, .497)],
    'wape_h1': 0.414, 'wape_h2': 0.482, 'degradacion': 0.165,
    'sesgo_h1': -0.076, 'sesgo_h2': -0.155, 'factor': 1.165,
}
_HCOL = 'horizonte'

print("═" * 92)
print("HORIZONTE — qué parte del entregable está ENCADENADA")
print("═" * 92)
print("""  horizonte 1 = el lag es un dato REAL          -> error ya conocido
  horizonte 2 = el lag es una PREDICCIÓN propia -> hereda el error del eslabón
  📌 No es distancia en el calendario: 202710 es h1 (se apoya en 202610, real)
     y 202720 es h2 (se apoya en 202620, que es proyectado).""")

# ── 1) EL PESO EN EL ENTREGABLE — la pregunta que faltaba ───────────────────
print("\n" + "─" * 92)
print("1 · CUÁNTO PESA CADA HORIZONTE")
print("─" * 92)
_g = proy.groupby(_HCOL).agg(
    filas=('P50', 'size'), periodos=('PERIODO', 'nunique'),
    p50=('P50', 'sum'), esperado=('P50_ESPERADO', 'sum'),
    secciones=('N_SECCIONES_TEORICAS_PROYECTADAS', 'sum'))
_g['%_volumen'] = _g['esperado'] / _g['esperado'].sum()
print(f"  {'h':<3} {'períodos':>9} {'filas':>8} {'Σ P50':>11} {'Σ esperado':>12} "
      f"{'%vol':>7} {'secciones':>10}")
for _h, _r in _g.iterrows():
    print(f"  {_h:<3} {_r['periodos']:>9,.0f} {_r['filas']:>8,.0f} {_r['p50']:>11,.0f} "
          f"{_r['esperado']:>12,.0f} {_r['%_volumen']:>7.1%} {_r['secciones']:>10,.0f}")

_ph = proy.groupby([_HCOL, 'PERIODO'])['P50_ESPERADO'].sum().unstack(0).fillna(0)
print(f"\n  qué período cae en cada horizonte:")
for _p, _r in _ph.iterrows():
    _h = 1 if _r.get(1, 0) > 0 else 2
    _marca = '' if _h == 1 else '   ⚠️ encadenado'
    print(f"    {_p}   h{_h}   {_r.sum():>9,.0f} inscritos esperados{_marca}")

_pct_h2 = _g.loc[2, '%_volumen'] if 2 in _g.index else 0.0

# ── 2) LO QUE CUESTA ENCADENAR — medido, no supuesto ───────────────────────
print("\n" + "─" * 92)
print("2 · LO QUE CUESTA ENCADENAR  (ML celda 22 · 4 pares reentrenados)")
print("─" * 92)
print(f"  {'par':<16} {'n':>7} {'WAPE h1':>9} {'WAPE h2':>9} {'degradación':>13}")
for _e, _n, _w1, _w2 in BT_H2['pares']:
    print(f"  {_e:<16} {_n:>7,} {_w1:>9.3f} {_w2:>9.3f} {_w2/_w1-1:>+13.1%}")
print("  " + "─" * 58)
print(f"  {'MEDIA':<16} {sum(x[1] for x in BT_H2['pares']):>7,} "
      f"{BT_H2['wape_h1']:>9.3f} {BT_H2['wape_h2']:>9.3f} "
      f"{BT_H2['degradacion']:>+13.1%}")
print(f"""
  ⚠️ Y EL SESGO NO SOLO CRECE: SE DUPLICA.
     sesgo h1 {BT_H2['sesgo_h1']:+.1%}  ->  h2 {BT_H2['sesgo_h2']:+.1%}
     El error de h1 no se cancela al encadenar — se arrastra y se acumula en el
     mismo sentido. Por eso el h2 sub-proyecta sistemáticamente.""")

# ── 3) GUARD — ¿la banda del h2 está realmente ensanchada? ─────────────────
## 🔧 CÓMO SE MIDE, que no es obvio. Comparar el ancho medio de h1 contra el de
##    h2 a secas NO sirve: son períodos distintos, con distinto mix de series y
##    distinto nivel de P50. Hay que controlar las dos cosas:
##      ① PAREAR por serie y por TT — 202720(h2) contra 202620(h1), mismo TT 20,
##        misma serie. Así el mix deja de contaminar.
##      ② NORMALIZAR por el cambio de nivel: en h2 el P50 baja, y una banda
##        proporcionalmente igual se ve más angosta en absoluto. El ensanchamiento
##        real es  (Σancho_h2 / Σancho_h1) ÷ (ΣP50_h2 / ΣP50_h1).
##    Se usan sumas y no medianas porque el ratio por fila es muy disperso.
print("\n" + "─" * 92)
print("3 · GUARD · ¿la banda del h2 está ensanchada como declara el backtest?")
print("─" * 92)
_PARES_TT = [(20, 202620, 202720), (25, 202625, 202725)]
_K4 = ['MATERIA', 'CURSO', 'COMPONENTE', 'CAMPUS']
_pr = proy.assign(_anc=proy['P90'] - proy['P10'])
print(f"  {'TT':<4} {'series':>8} {'Σancho h2/h1':>14} {'ΣP50 h2/h1':>12} "
      f"{'ensanche real':>14}")
_ens = []
for _tt, _a, _b in _PARES_TT:
    _A = _pr[_pr['PERIODO'] == _a].set_index(_K4)[['_anc', 'P50']]
    _B = _pr[_pr['PERIODO'] == _b].set_index(_K4)[['_anc', 'P50']]
    _j = _A.join(_B, how='inner', lsuffix='_1', rsuffix='_2').dropna()
    if not len(_j) or _j['_anc_1'].sum() <= 0 or _j['P50_1'].sum() <= 0:
        continue
    _ra = _j['_anc_2'].sum() / _j['_anc_1'].sum()
    _rp = _j['P50_2'].sum() / _j['P50_1'].sum()
    _e = _ra / _rp if _rp else np.nan
    _ens.append(_e)
    print(f"  {_tt:<4} {len(_j):>8,} {_ra:>14.3f} {_rp:>12.3f} {_e:>14.3f}")

_obs = float(np.mean(_ens)) if _ens else np.nan
print(f"\n  ensanchamiento EFECTIVO en el entregable : {_obs:.3f}")
print(f"  FACTOR_H2 que el backtest dice necesario : {BT_H2['factor']:.3f}")
if not np.isfinite(_obs):
    print("  ⚠️ no se pudo calcular — revisar P10/P90 o el pareo de períodos")
elif _obs >= BT_H2['factor'] - 0.03:
    print("  ✅ la banda del h2 está ensanchada como corresponde")
else:
    print(f"""  🚨 SUB-ENSANCHADA. Falta {BT_H2['factor'] - _obs:.3f} de ensanchamiento ({(BT_H2['factor']-_obs)/(BT_H2['factor']-1):.0%} del ajuste).

     El backtest mide que encadenar cuesta {BT_H2['degradacion']:.1%} más de error, y el
     `FACTOR_H2` existe para ensanchar la banda en esa proporción. En el
     entregable ese ensanchamiento llega solo a {_obs:.1%}-1 = {_obs-1:.1%}.

     CONSECUENCIA (T17 §9): la banda P10–P90 de 202720 y 202725 va a cubrir
     MENOS del 80% nominal. Un intervalo angosto mal calibrado da falsa
     seguridad — es peor que uno ancho honesto.

     ⚠️ CAUSA DIAGNOSTICADA (2026-09-01) — es un ERROR DE UNIDADES, no de orden.
        En el ML (celda 23, l.404) el factor se aplica así:

            _q_ancho = _Q * (_FACTOR_H2 if encadenado else 1.0)
            P10 = calibrar(..., -_q_ancho)   # shift en espacio LOG, antes de expm1
            P90 = calibrar(..., +_q_ancho)

        `_FACTOR_H2` mide cuánto sube el ERROR (WAPE ×1,165), pero `_Q` es un
        SHIFT LOGARÍTMICO del conformal (≈0,05–0,17). Multiplicar el shift por
        1,165 NO multiplica el ancho por 1,165: lo multiplica por ≈exp(Q×0,165),
        o sea +1% a +3%. Son dos unidades distintas.

        VERIFICADO: con Q≈0,255, exp(0,255×0,165) = 1,043 — exactamente el
        ensanchamiento observado. El reordenamiento de cuantiles NO tiene culpa.

        ARREGLO (requiere reentrenar; escalar el ancho DESPUÉS de reordenar):
            if encadenado:
                _fh['P10'] = _fh['P50'] - (_fh['P50'] - _fh['P10']) * _FACTOR_H2
                _fh['P90'] = _fh['P50'] + (_fh['P90'] - _fh['P50']) * _FACTOR_H2""")

# ── 4) QUÉ HACER CON ESTO ──────────────────────────────────────────────────
print("\n" + "═" * 92)
print("👔 LECTURA EJECUTIVA — no todo el entregable vale lo mismo")
print("═" * 92)
_deg_ins = _g.loc[2, 'esperado'] * BT_H2['degradacion'] if 2 in _g.index else 0
print(f"""  El {_pct_h2:.0%} del volumen proyectado ({_g.loc[2, 'esperado']:,.0f} inscritos en {_g.loc[2, 'periodos']:.0f} períodos)
  NO se apoya en ningún dato real: su lag es una predicción de este mismo modelo.

  Ese tramo trae {BT_H2['degradacion']:.0%} más de error que el resto —medido, no estimado— y
  sub-proyecta el doble ({BT_H2['sesgo_h1']:+.0%} -> {BT_H2['sesgo_h2']:+.0%}). En volumen, la banda de
  incertidumbre extra equivale a ~{_deg_ins:,.0f} inscritos.

  CÓMO USARLO AL PROGRAMAR:
  · Los {_g.loc[1, 'periodos']:.0f} períodos de h1 se usan como cualquier proyección del modelo.
  · Los {_g.loc[2, 'periodos']:.0f} de h2 (202720 y 202725) son PLANIFICACIÓN PRELIMINAR: sirven para
    dimensionar un orden de magnitud, no para comprometer secciones ni contratos.
  · Cuando 202620 y 202625 cierren con datos reales, esos dos períodos pasan a h1
    y deben re-proyectarse. Es la fecha en que el entregable se vuelve accionable.

  📌 Y como sub-proyectan sistemáticamente, si hay que equivocarse ahí conviene
     hacerlo por arriba: usar el P90 en vez del P50 para el tramo encadenado.""")

if np.isfinite(_obs) and _obs < BT_H2['factor'] - 0.03:
    print(f"""  🚨 Y HAY UNA RAZÓN EXTRA para no fiarse de la banda ahí: el guard de arriba
     detecta que el intervalo P10–P90 del h2 está ensanchado solo {_obs-1:.1%} cuando
     debería estarlo {BT_H2['factor']-1:.1%}. O sea que el tramo encadenado no solo tiene más
     error: además su banda dice tener menos incertidumbre de la que tiene.""")

metricas_horizonte = _g.copy()
print("\n  → disponible como `metricas_horizonte`")


════════════════════════════════════════════════════════════════════════════════════════════
HORIZONTE — qué parte del entregable está ENCADENADA
════════════════════════════════════════════════════════════════════════════════════════════
  horizonte 1 = el lag es un dato REAL          -> error ya conocido
  horizonte 2 = el lag es una PREDICCIÓN propia -> hereda el error del eslabón
  📌 No es distancia en el calendario: 202710 es h1 (se apoya en 202610, real)
     y 202720 es h2 (se apoya en 202620, que es proyectado).

────────────────────────────────────────────────────────────────────────────────────────────
1 · CUÁNTO PESA CADA HORIZONTE
────────────────────────────────────────────────────────────────────────────────────────────
  h    períodos    filas       Σ P50   Σ esperado    %vol  secciones
  1           5   15,419     811,161      736,078   71.7%     36,266
  2           2    6,723     315,117      290,455   28.3%     15,188

  qué período cae en cada horizonte:
    202620 

In [6]:
## ══════════════════════════════════════════════════════════════════════════════
## NIVELES DE DESAGREGACIÓN — de MACRO a MICRO, con su margen de error
## ══════════════════════════════════════════════════════════════════════════════
## Audiencia: 🔧 🧭 👔
##
## La tabla baja por la jerarquía agregando una dimensión a la vez, hasta llegar
## al grano en que el modelo realmente predice:
##
##                              ┌ +Componente ┐
##     Período → +Campus →      ┤             ├→ grano del modelo
##                              └ +Materia×Curso ┘
##
## ⚠️ EMPIEZA EN PERÍODO, no en año ni en el total institucional. Es una decisión
##    de negocio, no técnica: la programación académica SIEMPRE se hace para un
##    período. Un número anual no corresponde a ninguna decisión que alguien tome.
##    📌 Se midió igual, y quedó en la bitácora: al corte ANUAL el modelo PIERDE
##       contra el naive (skill −8,1%). Que no aparezca en la tabla no significa
##       que ahí funcione — significa que ahí no se decide nada.
##
## ⚠️ Los dos niveles del medio son HERMANOS, no padre-hijo: se llega al grano del
##    modelo por cualquiera de los dos caminos. Se muestran los dos porque tienen
##    tamaños muy distintos (~1.400 vs ~40 estudiantes por celda) y por tanto
##    márgenes de error muy distintos — es justo el rango donde se programa.
##
## Al ir bajando, lo único que cambia es cuánto se está agregando. Cualquier
## mejora de métrica hacia arriba es COMPENSACIÓN de errores, no mérito del modelo.
##
## ── LAS DOS PREGUNTAS QUE RESPONDE ────────────────────────────────────────────
##
## ① ¿APORTA el modelo a este nivel?  ->  SKILL SCORE
##
##       SS = 1 − (error_modelo / error_referencia)
##
##   SS = 0    empata con el naive — no aporta nada
##   SS > 0    le gana; 0,30 = reduce el error del naive en un 30%
##   SS < 0    PIERDE contra "igual que el año pasado"
##
##   🔧 Se reporta así y no con el WAPE suelto porque el WAPE depende del universo
##      de series y del nivel de agregación: no es comparable entre corridas ni
##      entre niveles. El skill normaliza contra la vara de ESE mismo corte.
##      (La Guía Maestra T17 §10 usa el concepto sin nombrarlo; es el mismo cociente.)
##
## ② ¿CUÁNTO ME PUEDO DESVIAR proyectando N estudiantes?  ->  MARGEN DE ERROR
##
##   Distribución EMPÍRICA del error en los períodos ya cerrados. No es un
##   intervalo teórico ni un supuesto de normalidad: es lo que efectivamente pasó.
##
## ⚠️ EL WAPE BAJA AL AGREGAR, y no porque el modelo mejore: los errores de signo
##    opuesto se cancelan. Un WAPE institucional de 0,04 no significa "4% de error"
##    en nada que se decida — significa que los +130% de las series chicas taparon
##    los −14% de las grandes. Por eso la tabla termina en el grano de decisión.
##
## 📌 Todos los niveles llevan el período dentro. Los cortes que agregan sobre todo
##    el tiempo (Campus a secas, Campus × Componente) se midieron y quedaron en la
##    bitácora: ahí el modelo PIERDE contra el naive (−14,0% y −12,7%). No se
##    muestran porque una celda que mezcla 23 períodos no es una decisión.
## ──────────────────────────────────────────────────────────────────────────────
_b = hist[hist['INSCRITOS_REAL'].notna() & hist['PROY_ESTANDAR'].notna()].copy()
_b = _b.merge(pd.read_parquet(f'{CARPETA}/DEMANDA_PROCESADA.parquet',
                              columns=DIMS + ['ins_lag_1y']), on=DIMS, how='left')
_b = _b[_b['ins_lag_1y'].notna()]      # sólo donde la vara existe: comparación justa

# ── LOS NIVELES: (nombre, dimensiones, sangría del árbol) ────────────────────
# El último nivel es exactamente DIMS: el grano en que el modelo predice.
_PER_CAM = ['PERIODO', 'CAMPUS']
_NIVELES = [
    ('Período',                    ['PERIODO'],                       ''),
    ('… × Campus',                 _PER_CAM,                          '  └ '),
    # ── dos ramas hermanas: mismo padre, mismo destino, tamaños muy distintos ──
    ('… × Componente',             _PER_CAM + ['COMPONENTE'],         '    ├ '),
    ('… × Materia × Curso',        _PER_CAM + ['MATERIA', 'CURSO'],   '    ├ '),
    ('… × Materia×Curso × Comp.',  _PER_CAM + ['MATERIA', 'CURSO',
                                               'COMPONENTE'],         '    └ '),
]
_GRANO_MODELO = '… × Materia×Curso × Comp.'
assert sorted(_NIVELES[-1][1]) == sorted(DIMS), 'el último nivel debe ser DIMS'

print("═" * 92)
print("NIVELES DE DESAGREGACIÓN — de MACRO a MICRO")
print("═" * 92)
print(f"  {len(_b):,} filas · {_b['PERIODO'].nunique()} períodos cerrados · "
      f"{_b['INSCRITOS_REAL'].sum():,.0f} inscritos")
print("  WAPE y SKILL se calculan sobre los valores YA AGREGADOS a cada nivel,")
print("  no promediando los de abajo — así se ve la compensación real.")
print("  Arranca en PERÍODO: es el horizonte en que se programa. El corte anual")
print("  existe pero no se decide nada en él (y ahí el modelo pierde: −8,1%).\n")


def _wape(y, f):
    y, f = np.asarray(y, float), np.asarray(f, float)
    return np.abs(f - y).sum() / np.abs(y).sum() if np.abs(y).sum() else np.nan


def _mide(nombre, by, pref):
    """Métricas de un nivel: aporte (skill) y margen de error (distribución)."""
    g = _b.groupby(by, observed=True)[['INSCRITOS_REAL', 'PROY_ESTANDAR',
                                       'ins_lag_1y']].sum()
    wm = _wape(g['INSCRITOS_REAL'], g['PROY_ESTANDAR'])
    wn = _wape(g['INSCRITOS_REAL'], g['ins_lag_1y'])

    # ── MARGEN DE ERROR ─────────────────────────────────────────────────────
    # La unidad de observación es UNA celda del nivel = UNA proyección concreta.
    # Se mide sobre las MISMAS celdas que el skill, sin forzar ninguna dimensión
    # extra: lo que se reporta es el error de esa proyección, tal cual se hace.
    gp = g[g['INSCRITOS_REAL'] > 0]
    err = gp['PROY_ESTANDAR'] / gp['INSCRITOS_REAL'] - 1        # error relativo
    return {'nivel': nombre, 'pref': pref, 'celdas': len(g), 'obs': len(gp),
            'wape_modelo': wm, 'wape_naive': wn,
            'skill': 1 - wm / wn if wn else np.nan,
            # tamaño típico de UNA proyección: es el puente entre "estoy
            # proyectando N estudiantes" y qué fila de la tabla me aplica
            'n_tipico': gp['INSCRITOS_REAL'].median(),
            'err_p50': err.abs().median(), 'err_p90': err.abs().quantile(.90),
            'lo': err.quantile(.05), 'hi': err.quantile(.95)}


def _avisos(r):
    """Marcas al FINAL de la línea: los emojis ocupan dos columnas y romperían
    el ancho fijo de cualquier campo intermedio."""
    a = ' ⚠️PIERDE' if r['skill'] <= 0 else ''
    # con menos de 30 celdas el p90 es casi el máximo: no es un percentil estable
    a += ' ·muestra corta' if r['obs'] < 30 else ''
    return a + ('  ⭐ GRANO DEL MODELO' if r['nivel'] == _GRANO_MODELO else '')


# el total institucional no es fila (una sola celda), pero se cita en la lectura
_WAPE_INST = _wape([_b['INSCRITOS_REAL'].sum()], [_b['PROY_ESTANDAR'].sum()])

skill_por_nivel = pd.DataFrame([_mide(n, b, p) for n, b, p in _NIVELES])

print(f"  {'nivel':<34} {'celdas':>7} {'N típico':>9} {'WAPE':>6} {'SKILL':>7} | "
      f"{'error típ.':>10} {'≤9 de 10':>9} {'rango 90%':>18}")
print("  " + "─" * 112)
for _, _r in skill_por_nivel.iterrows():
    print(f"  {_r['pref'] + _r['nivel']:<34} {_r['celdas']:>7,} {_r['n_tipico']:>9,.0f} "
          f"{_r['wape_modelo']:>6.3f} {_r['skill']:>+7.1%} | "
          f"{_r['err_p50']:>9.1%} {_r['err_p90']:>9.1%} "
          f"{'{:+.0%} a {:+.0%}'.format(_r['lo'], _r['hi']):>18}{_avisos(_r)}")

print("""
  ── CÓMO SE LEE EL MARGEN DE ERROR ───────────────────────────────────────────
   «¿con qué % de error me puedo encontrar proyectando N estudiantes?»

   · «N típico»    cuántos estudiantes tiene UNA celda de este nivel (mediana).
                   ⭐ ES LA COLUMNA DE ENTRADA: busca la fila cuyo N se parezca
                      al volumen que vas a proyectar y lee su error a la derecha.
   · «error típ.»  mediana del |error %| — la mitad de las veces te desvías menos.
   · «≤9 de 10»    el error NO supera esto en el 90% de los casos. Éste es el
                   número para dimensionar el colchón, no el «error típico».
   · «rango 90%»   entre qué dos valores cae el error CON SIGNO. Si es asimétrico
                   el modelo se equivoca más hacia un lado, y conviene saber cuál.

   ⚠️ «muestra corta» = menos de 30 celdas; ahí el p90 es casi el máximo
      observado, no un percentil estable. Léelo como orden de magnitud.
   ⚠️ Las dos filas con «├» son HERMANAS: se baja del campus al grano del modelo
      por cualquiera de las dos. No se suman ni se encadenan entre sí.""")

_mac, _mic = skill_por_nivel.iloc[0], skill_por_nivel.iloc[-1]
_camp = skill_por_nivel[skill_por_nivel['nivel'] == '… × Campus'].iloc[0]
_cc = skill_por_nivel[skill_por_nivel['nivel'] == '… × Componente'].iloc[0]
_mc = skill_por_nivel[skill_por_nivel['nivel'] == '… × Materia × Curso'].iloc[0]
print(f"""
  ── QUÉ MUESTRA LA TABLA ─────────────────────────────────────────────────────
   · El WAPE va de {_WAPE_INST:.3f} (total institucional, fuera de tabla) y {_mac['wape_modelo']:.3f}
     (período) hasta {_mic['wape_modelo']:.3f} en el grano del modelo. Esa caída NO es mérito
     del modelo: es la compensación de errores de signo opuesto al sumar.
   · El error típico sube de {_mac['err_p50']:.1%} a {_mic['err_p50']:.1%} al bajar por la jerarquía.
     Es el MISMO modelo: lo que cambia es cuánto se está agregando.
   · LAS DOS RAMAS: bajar por componente deja celdas de ~{_cc['n_tipico']:,.0f} estudiantes con
     {_cc['err_p50']:.0%} de error típico; bajar por asignatura deja celdas de ~{_mc['n_tipico']:,.0f} con {_mc['err_p50']:.0%}.
     Es el mismo modelo y el mismo campus: cambia con qué grano se pregunta.
   · El SKILL es lo comparable entre filas, porque cada una se mide contra SU
     propia vara — el naive de ese mismo corte.""")

_pierde = skill_por_nivel[skill_por_nivel['skill'] <= 0]['nivel'].tolist()
print(f"""
   ⚠️ El modelo NO le gana al naive en: {', '.join(_pierde)}
      A ese corte conviene usar el naive directamente: es gratis y acierta más."""
      if _pierde else """
   ✅ El modelo le gana al naive en TODOS los niveles de programación.
      (Al corte anual sí pierde, −8,1%; pero ahí no se decide nada — por eso
       la tabla arranca en período.)""")

print("\n" + "═" * 92)
print("👔 LECTURA EJECUTIVA — cuánto confiar en el número")
print("═" * 92)
print(f"""  Cuanto más agregado el número, más pequeño parece el error — y menos
  significa. Al nivel al que se DECIDE, el modelo reduce el error del "igual que
  el año pasado" en un {_mic['skill']:.0%}. Ése, y no el {_WAPE_INST:.1%} institucional, es el número honesto.

  CUÁNTO CONFIAR, SEGÚN LO QUE ESTÉS PROYECTANDO:
  · Un PERÍODO completo (~{_mac['n_tipico']:,.0f} estudiantes): desvío típico {_mac['err_p50']:.0%}, rara vez
    sobre {_mac['err_p90']:.0%}. Sirve para dimensionar el presupuesto del período.
  · Una SEDE en ese período (~{_camp['n_tipico']:,.0f} estudiantes): desvío típico {_camp['err_p50']:.0%}, hasta
    {_camp['err_p90']:.0%} una de cada diez veces. Sirve para planificar carga docente.
  · Un COMPONENTE de esa sede (~{_cc['n_tipico']:,.0f} estudiantes): desvío típico {_cc['err_p50']:.0%}, hasta
    {_cc['err_p90']:.0%}. ⭐ Es el grano al que se dimensiona la SALA y el tipo de espacio.
  · Una ASIGNATURA-COMPONENTE en una sede (~{_mic['n_tipico']:,.0f} estudiantes): desvío típico
    {_mic['err_p50']:.0%}, y una de cada diez se va sobre {_mic['err_p90']:.0%}. Ahí el número NO se toma
    solo: se toma con la banda P10–P90 y criterio de la sede.

  📌 La regla operativa: mientras más fina la decisión, más pesa el criterio
     humano sobre el modelo. El modelo ordena la conversación, no la reemplaza.""")

print("\n  → disponible como `skill_por_nivel`")


════════════════════════════════════════════════════════════════════════════════════════════
NIVELES DE DESAGREGACIÓN — de MACRO a MICRO
════════════════════════════════════════════════════════════════════════════════════════════
  55,983 filas · 23 períodos cerrados · 2,612,770 inscritos
  WAPE y SKILL se calculan sobre los valores YA AGREGADOS a cada nivel,
  no promediando los de abajo — así se ve la compensación real.
  Arranca en PERÍODO: es el horizonte en que se programa. El corte anual
  existe pero no se decide nada en él (y ahí el modelo pierde: −8,1%).

  nivel                               celdas  N típico   WAPE   SKILL | error típ.  ≤9 de 10          rango 90%
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Período                                 23    44,250  0.063  +17.7% |      6.2%     16.4%         -8% a +20% ·muestra corta
    └ … × Campus                         132     9,888  0.086   +5.4% |     

In [7]:
## ══════════════════════════════════════════════════════════════════════════════
## MÉTRICAS DE LA CONVERSIÓN A SECCIONES — global (T08 §5 · T17 §10)
## ══════════════════════════════════════════════════════════════════════════════
## Audiencia: 🔧 🧭 👔
##
## QUÉ MIDE: no el modelo de demanda, sino el paso siguiente — convertir inscritos
##   en SECCIONES. Es una cadena de dos eslabones, y cada uno aporta error:
##
##       inscritos reales ──[modelo]──> P50 ──[estándar ÷ 1,1]──> nº de secciones
##                              ▲                    ▲
##                       error del MODELO      error del ESTÁNDAR
##
## ── LA VARA: «Norma·real» (N_SECCIONES_TEORICAS) ──────────────────────────────
##   La MISMA norma aplicada a los inscritos REALES: lo que la conversión habría
##   dado con una proyección perfecta.
##
##   🔧 POR QUÉ ÉSTA: las secciones reales no siguen el estándar teórico, así que
##      medir contra "las mismas del año pasado" mezcla el desajuste de LA NORMA
##      con el error de LA PROYECCIÓN — y el primero domina. Con esta vara la
##      norma queda en los dos lados del cociente y se cancela, dejando visible
##      lo único que el modelo de demanda puede mover.
##
##   ⚠️ La vara NO es perfecta ni es un techo: arrastra el desajuste del estándar,
##      así que un criterio con mejor denominador puede ganarle.
##
## ⚠️ LAS SECCIONES SON ENTEROS PEQUEÑOS: la mediana por fila es 1 y el 75% tiene
##    ≤2. Con denominadores así el error PORCENTUAL es engañoso — fallar por una
##    sección sobre una es "100% de error". Por eso aquí manda el error ABSOLUTO
##    (cuántas secciones de más o de menos) y el % de aciertos exactos.
## ──────────────────────────────────────────────────────────────────────────────
_S4 = ['MATERIA', 'CURSO', 'COMPONENTE', 'CAMPUS']
_SEC_REAL = 'N_SECCIONES_REALES'
_VARA, _VARA_ETQ = 'N_SECCIONES_TEORICAS', 'Norma·real  (norma + inscritos REALES)'
_SEC_CRIT = [
    ('N_SECCIONES_TEORICAS_PROYECTADAS', '⭐ Norma·P50  (el operativo)'),
    ('N_SECCIONES_PROMEDIO',             'Ocupación observada · P50'),
    ('N_SECCIONES_P25',                  'escenario p25 · P50'),
    ('N_SECCIONES_P50',                  'escenario p50 · P50'),
    ('N_SECCIONES_P75',                  'escenario p75 · P50'),
    ('N_SECCIONES_P90',                  'escenario p90 · P50'),
]

_sb = hist[hist[_SEC_REAL] > 0].copy()
_sr = _sb[_SEC_REAL]

print("═" * 90)
print("MÉTRICAS DE LA CONVERSIÓN A SECCIONES")
print("═" * 90)
print(f"  {len(_sb):,} filas · {_sb['PERIODO'].nunique()} períodos cerrados · "
      f"{_sr.sum():,.0f} secciones reales")


def _wape_s(y, f):
    y, f = np.asarray(y, float), np.asarray(f, float)
    return np.abs(f - y).sum() / np.abs(y).sum() if np.abs(y).sum() else np.nan


# ── 1) LA VARA primero, como manda la regla de oro nº1 (T17 §10) ─────────────
print("\n" + "─" * 90)
print("1 · LA VARA — la norma aplicada a los inscritos REALES")
print("─" * 90)
_wv = _wape_s(_sr, _sb[_VARA])
_dv = _sb[_VARA] - _sr
print(f"  Σ secciones de la vara : {_sb[_VARA].sum():>9,.0f}  "
      f"({_sb[_VARA].sum() / _sr.sum() - 1:+.1%} vs real)")
print(f"  WAPE de la vara        : {_wv:>9.3f}   ← la vara a batir")
print(f"  acierta exacto         : {(_dv == 0).mean():>9.1%}")
print(f"""
  ⚠️ Ese {_sb[_VARA].sum() / _sr.sum() - 1:+.1%} es el TECHO DEL ESTÁNDAR: aun conociendo la demanda
     exacta, la norma abre esa proporción de secciones de menos. Ninguna mejora
     del modelo de demanda lo toca.""")

# ── 1b) Referencia externa: el naive estacional, sobre su propia base ───────
_lag = hist[_S4 + ['PERIODO', _SEC_REAL]].copy()
_lag['PERIODO'] += 100
_lag = _lag.rename(columns={_SEC_REAL: 'sec_lag_1y'})
_nv = hist.merge(_lag, on=_S4 + ['PERIODO'], how='left')
_nv = _nv[_nv['sec_lag_1y'].notna() & (_nv[_SEC_REAL] > 0)]
_wn_est = _wape_s(_nv[_SEC_REAL], _nv['sec_lag_1y'])
print(f"""  📌 REFERENCIA EXTERNA (no es la vara del skill de abajo): repetir las
     secciones del mismo TT del año anterior da WAPE {_wn_est:.3f} y acierta el
     {(_nv['sec_lag_1y'] == _nv[_SEC_REAL]).mean():.1%} exacto — mejor que cualquier criterio de conversión.
     ⚠️ Base distinta ({len(_nv):,} filas, {_nv['PERIODO'].nunique()} períodos): 2022 no tiene año previo.
     Se reporta porque el hallazgo importa, no para compararlo columna a columna.""")

# ── 2) CADA CRITERIO CONTRA LA VARA ─────────────────────────────────────────
print("\n" + "─" * 90)
print("2 · LOS SEIS CRITERIOS DE CONVERSIÓN")
print("─" * 90)
print(f"  {'criterio':<32} {'Σ sec':>9} {'vs real':>8} {'WAPE':>6} {'SKILL':>7} "
      f"{'exacta':>7} {'±1 sec':>7} {'p90 |err|':>9}")
print("  " + "─" * 90)
_filas_s = []
for _c, _e in _SEC_CRIT:
    _d = _sb[_c] - _sr
    _w = _wape_s(_sr, _sb[_c])
    _filas_s.append({'criterio': _c, 'etiqueta': _e, 'suma': _sb[_c].sum(),
                     'sesgo': _sb[_c].sum() / _sr.sum() - 1, 'wape': _w,
                     'skill': 1 - _w / _wv, 'exacta': (_d == 0).mean(),
                     'mas_menos_1': (_d.abs() <= 1).mean(),
                     'err_abs_p90': _d.abs().quantile(.90)})
    _r = _filas_s[-1]
    print(f"  {_e:<32} {_r['suma']:>9,.0f} {_r['sesgo']:>+8.1%} {_r['wape']:>6.3f} "
          f"{_r['skill']:>+7.1%} {_r['exacta']:>7.1%} {_r['mas_menos_1']:>7.1%} "
          f"{_r['err_abs_p90']:>9.0f}" + ('  ⭐' if _r['skill'] > 0 else ''))
metricas_secciones = pd.DataFrame(_filas_s)

# ── 3) DESCOMPOSICIÓN: cuánto es del estándar y cuánto del modelo ───────────
_op = metricas_secciones.iloc[0]                 # el operativo: norma sobre P50
print("\n" + "─" * 90)
print("3 · DE QUIÉN ES EL ERROR — estándar vs modelo")
print("─" * 90)
print(f"  {'eslabón':<44} {'WAPE':>7} {'exacta':>8} {'sesgo':>9}")
print(f"  {'sólo el ESTÁNDAR (vara: inscritos reales)':<44} {_wv:>7.3f} "
      f"{(_dv == 0).mean():>8.1%} {_sb[_VARA].sum() / _sr.sum() - 1:>+9.1%}")
print(f"  {'ESTÁNDAR + MODELO (con el P50)':<44} {_op['wape']:>7.3f} "
      f"{_op['exacta']:>8.1%} {_op['sesgo']:>+9.1%}")
print(f"  {'diferencia = COSTO DEL MODELO':<44} {_op['wape'] - _wv:>+7.3f} "
      f"{_op['exacta'] - (_dv == 0).mean():>+8.1%} "
      f"{_op['sesgo'] - (_sb[_VARA].sum() / _sr.sum() - 1):>+9.1%}")
print(f"""
  ⚠️ OJO CON EL SIGNO DEL SESGO: el modelo EMPEORA la precisión (WAPE {_op['wape'] - _wv:+.3f},
     exactas {_op['exacta'] - (_dv == 0).mean():+.1%}) pero MEJORA el sesgo agregado ({_sb[_VARA].sum()/_sr.sum()-1:+.1%} -> {_op['sesgo']:+.1%}).
     No es una virtud: el modelo sobre-proyecta inscritos y ese sesgo positivo
     cancela parcialmente el sub-dimensionamiento de la norma. Dos errores que
     se compensan, no uno que se corrige. Si mañana se arregla el estándar, ese
     sesgo del modelo quedaría al descubierto — hay que arreglar los dos.""")

# ── 4) DÓNDE FALLA — el error no es uniforme por tamaño ─────────────────────
print("\n" + "─" * 90)
print("4 · DÓNDE SE CONCENTRA EL ERROR  (🧭 T08 §6: la métrica global esconde)")
print("─" * 90)
_sb['_tr'] = pd.cut(_sr, [0, 1, 3, 10, np.inf], labels=['1', '2-3', '4-10', '>10'])
print(f"  {'secciones':<10} {'filas':>8} {'%sec':>7} {'WAPE':>7} {'sesgo':>8} "
      f"{'exacta':>8} {'±1 sec':>8}   {'← el mismo corte para la VARA':>0}")
for _k, _x in _sb.groupby('_tr', observed=True):
    _d = _x['N_SECCIONES_TEORICAS_PROYECTADAS'] - _x[_SEC_REAL]
    _dv2 = _x[_VARA] - _x[_SEC_REAL]
    print(f"  {_k:<10} {len(_x):>8,} {_x[_SEC_REAL].sum()/_sr.sum():>6.1%} "
          f"{_wape_s(_x[_SEC_REAL], _x['N_SECCIONES_TEORICAS_PROYECTADAS']):>7.3f} "
          f"{_d.sum()/_x[_SEC_REAL].sum():>+8.1%} {(_d == 0).mean():>8.1%} "
          f"{(_d.abs() <= 1).mean():>8.1%}   vara: sesgo "
          f"{_dv2.sum()/_x[_SEC_REAL].sum():>+6.1%}")
print("""
  📌 Las filas de UNA sección son la mayoría y casi siempre se aciertan: ahí la
     decisión es binaria (se abre o no), no de dimensionamiento. El error vive en
     las asignaturas grandes, que son pocas pero pesan — y ahí el sesgo de la
     VARA muestra que el problema ya está en la norma, antes de proyectar nada.""")

print("\n" + "═" * 90)
print("👔 EN UNA FRASE PARA EL NEGOCIO")
print("═" * 90)
print(f"""  Convertir la proyección en secciones acierta el número exacto en el {_op['exacta']:.0%} de
  los casos y queda a ±1 sección en el {_op['mas_menos_1']:.0%}. Pero abre un {abs(_op['sesgo']):.0%} menos de
  secciones que las que se abrieron, y ese hueco NO es del modelo: con los
  inscritos verdaderos la norma ya abre un {abs(_sb[_VARA].sum()/_sr.sum()-1):.0%} de menos. La palanca está
  en revisar el estándar, no en afinar la proyección.""")

print("\n  → disponible como `metricas_secciones`")


══════════════════════════════════════════════════════════════════════════════════════════
MÉTRICAS DE LA CONVERSIÓN A SECCIONES
══════════════════════════════════════════════════════════════════════════════════════════
  55,983 filas · 23 períodos cerrados · 168,974 secciones reales

──────────────────────────────────────────────────────────────────────────────────────────
1 · LA VARA — la norma aplicada a los inscritos REALES
──────────────────────────────────────────────────────────────────────────────────────────
  Σ secciones de la vara :   123,335  (-27.0% vs real)
  WAPE de la vara        :     0.360   ← la vara a batir
  acierta exacto         :     71.9%

  ⚠️ Ese -27.0% es el TECHO DEL ESTÁNDAR: aun conociendo la demanda
     exacta, la norma abre esa proporción de secciones de menos. Ninguna mejora
     del modelo de demanda lo toca.
  📌 REFERENCIA EXTERNA (no es la vara del skill de abajo): repetir las
     secciones del mismo TT del año anterior da WAPE 0.289 y acierta el


In [8]:
## ══════════════════════════════════════════════════════════════════════════════
## SECCIONES · CADA CRITERIO EN CADA NIVEL — la misma comparativa del gráfico
## ══════════════════════════════════════════════════════════════════════════════
## Audiencia: 🔧 🧭 👔
##
## Es la tabla del gráfico («Reales · Norma·real · Norma·P50 · p25 · p50 · p75 ·
## p90») abierta por NIVEL DE AGREGACIÓN, en vez de por período.
##
##                              ┌ +Componente ┐
##     Período → +Campus →      ┤             ├→ grano del modelo
##                              └ +Materia×Curso ┘
##
## ── LA VARA: «Norma·real» ─────────────────────────────────────────────────────
##   N_SECCIONES_TEORICAS = la MISMA norma, aplicada a los inscritos REALES.
##   Es decir: lo que la conversión habría dado con una proyección perfecta.
##
##   🔧 POR QUÉ ÉSTA Y NO EL NAIVE ESTACIONAL: las secciones reales no siguen el
##      estándar teórico, así que medir contra "las mismas del año pasado" mezcla
##      dos cosas — el desajuste de LA NORMA y el error de LA PROYECCIÓN — y el
##      primero domina (−27,8% de desvío ya con inscritos verdaderos). Con esta
##      vara la norma queda en los dos lados del cociente y se cancela.
##
##   ⚠️ CÓMO SE LEE, QUE NO ES LO MISMO PARA TODAS LAS FILAS:
##      · «Norma·P50» vs la vara -> misma fórmula, sólo cambia la demanda.
##                                  El skill aísla EL COSTO DEL MODELO. Limpio.
##      · los percentiles vs la vara -> cambia el DENOMINADOR además de la
##                                  demanda. El skill mide el efecto combinado
##                                  de recalibrar el estándar contra la práctica.
##      SS = 0 -> igual que aplicar la norma con demanda perfecta.
##      SS > 0 -> LE GANA a la norma-con-demanda-perfecta: ese denominador
##                describe mejor lo que realmente se abre.
##
##   📌 Medido aparte y ya registrado: contra el naive estacional (las secciones
##      del mismo TT del año anterior) NINGÚN criterio gana, ni siquiera esta
##      vara. El paso inscritos → secciones es el eslabón débil, no la demanda.
##
## ⚠️ EL DESVÍO TOTAL NO DEPENDE DEL NIVEL. Σpred/Σreal − 1 da lo mismo se agrupe
##    como se agrupe: las sumas son las mismas. Por eso va como TABLA A (un valor
##    por criterio) y no como matriz — cinco columnas repetirían la misma cifra.
##
## ⚠️ LAS SECCIONES SON ENTEROS PEQUEÑOS (mediana 1 por fila, 75% ≤2). Por eso al
##    grano fino el % engaña y manda «exacta»: qué fracción de celdas da el número
##    justo. Arriba manda el %, abajo manda «exacta». Las matrices lo cruzan.
## ──────────────────────────────────────────────────────────────────────────────
_SEC_REAL = 'N_SECCIONES_REALES'
_VARA, _VARA_ETQ = 'N_SECCIONES_TEORICAS', 'Norma·real  (norma + inscritos REALES)'

# los criterios evaluados: los del gráfico MENOS la vara, que ya no compite
_CRIT = [
    ('N_SECCIONES_TEORICAS_PROYECTADAS', '⭐ Norma·P50  (el operativo)'),
    ('N_SECCIONES_PROMEDIO',             'Ocupación observada · P50'),
    ('N_SECCIONES_P25',                  'p25 · P50'),
    ('N_SECCIONES_P50',                  'p50 · P50'),
    ('N_SECCIONES_P75',                  'p75 · P50'),
    ('N_SECCIONES_P90',                  'p90 · P50'),
]
_PER_CAM = ['PERIODO', 'CAMPUS']
_NIV_S = [
    ('Período',        ['PERIODO']),
    ('+Campus',        _PER_CAM),
    ('+Compon.',       _PER_CAM + ['COMPONENTE']),
    ('+Mat×Curso',     _PER_CAM + ['MATERIA', 'CURSO']),
    ('⭐ grano',        _PER_CAM + ['MATERIA', 'CURSO', 'COMPONENTE']),
]
assert sorted(_NIV_S[-1][1]) == sorted(DIMS), 'el último nivel debe ser DIMS'

# 📌 La vara vive en el propio dataset (100% de cobertura en histórico), así que
#    aquí NO se pierde ninguna fila. El naive estacional obligaba a descartar
#    2022 entero — el 37% de las filas — por no tener año previo.
_n = hist[hist[_SEC_REAL] > 0].copy()
_TOT_REAL = _n[_SEC_REAL].sum()

print("═" * 100)
print("SECCIONES · CADA CRITERIO EN CADA NIVEL DE AGREGACIÓN")
print("═" * 100)
print(f"  {len(_n):,} filas · {_n['PERIODO'].nunique()} períodos cerrados · "
      f"{_TOT_REAL:,.0f} secciones reales")
print(f"  vara = {_VARA} · la misma norma con los inscritos REALES")


def _wape_n(y, f):
    y, f = np.asarray(y, float), np.asarray(f, float)
    return np.abs(f - y).sum() / np.abs(y).sum() if np.abs(y).sum() else np.nan


# ── TABLA A · el desvío total, que NO depende del nivel ─────────────────────
print("\n" + "─" * 100)
print("TABLA A · DESVÍO TOTAL vs SECCIONES REALES  —  la comparativa del gráfico")
print("─" * 100)
print("  Un valor por criterio: agrupar no lo cambia, porque las sumas son las mismas.")
print("  Negativo = abre de MENOS que lo que efectivamente se abrió.\n")
print(f"  {'criterio':<38} {'Σ secciones':>12} {'vs real':>10}")
print(f"  {'REAL — lo que se abrió':<38} {_TOT_REAL:>12,.0f}")
print(f"  {'⚖️ ' + _VARA_ETQ:<38} {_n[_VARA].sum():>12,.0f} "
      f"{_n[_VARA].sum() / _TOT_REAL - 1:>+10.1%}   ← la vara")
print("  " + "─" * 62)
for _c, _e in _CRIT:
    print(f"  {_e:<38} {_n[_c].sum():>12,.0f} {_n[_c].sum() / _TOT_REAL - 1:>+10.1%}")

# ── una fila por (nivel × criterio) ──────────────────────────────────────────
_filas, _ctx = [], []
for _nom, _by in _NIV_S:
    _g = _n.groupby(_by, observed=True)[[_SEC_REAL, _VARA] +
                                        [c for c, _ in _CRIT]].sum()
    _g = _g[_g[_SEC_REAL] > 0]
    _wv = _wape_n(_g[_SEC_REAL], _g[_VARA])
    _dv = _g[_VARA] - _g[_SEC_REAL]
    _ctx.append({'nivel': _nom, 'celdas': len(_g),
                 'n_tipico': _g[_SEC_REAL].median(), 'wape_vara': _wv,
                 'err_vara': (_dv / _g[_SEC_REAL]).abs().median(),
                 'exacta_vara': (_dv == 0).mean()})
    for _c, _e in _CRIT:
        _dif = _g[_c] - _g[_SEC_REAL]              # error en SECCIONES, con signo
        _w = _wape_n(_g[_SEC_REAL], _g[_c])
        _filas.append({
            'nivel': _nom, 'criterio': _e, 'columna': _c, 'celdas': len(_g),
            'suma': _g[_c].sum(), 'sesgo': _dif.sum() / _g[_SEC_REAL].sum(),
            'wape': _w, 'skill': 1 - _w / _wv if _wv else np.nan,
            'err_p50': (_dif / _g[_SEC_REAL]).abs().median(),
            'sec_p50': _dif.abs().median(), 'exacta': (_dif == 0).mean()})
secciones_por_nivel = pd.DataFrame(_filas)
contexto_niveles = pd.DataFrame(_ctx)

# ── contexto: cómo se comporta la VARA en cada nivel ────────────────────────
print("\n" + "─" * 100)
print("LA VARA EN CADA NIVEL  ·  el skill de abajo se mide contra estas cifras")
print("─" * 100)
print(f"  {'nivel':<14} {'celdas':>8} {'N típ. (sec)':>13} {'WAPE vara':>11} "
      f"{'err típ.':>10} {'exacta':>9}")
for _, _r in contexto_niveles.iterrows():
    print(f"  {_r['nivel']:<14} {_r['celdas']:>8,} {_r['n_tipico']:>13,.0f} "
          f"{_r['wape_vara']:>11.3f} {_r['err_vara']:>10.1%} {_r['exacta_vara']:>9.1%}")
print("""
  📌 La vara NO es perfecta: aunque use los inscritos verdaderos, arrastra el
     desajuste del estándar. Por eso un criterio PUEDE ganarle — le basta con
     tener un denominador más parecido a cómo se abren las secciones de verdad.""")

_NIVS = [n for n, _ in _NIV_S]
_ETQ = [e for _, e in _CRIT]


def _matriz(campo, titulo, nota, fmt, mejor='max'):
    """Matriz criterio × nivel. El ▸ marca el mejor criterio de cada columna."""
    print("\n" + "─" * 100)
    print(titulo)
    print("─" * 100)
    print(f"  {nota}\n")
    _p = secciones_por_nivel.pivot(index='criterio', columns='nivel',
                                   values=campo).reindex(_ETQ)[_NIVS]
    _win = {c: (_p[c].abs().idxmin() if mejor == 'cero' else _p[c].idxmax())
            for c in _NIVS}
    print(f"  {'criterio':<32}" + ''.join(f'{c:>13}' for c in _NIVS))
    print("  " + "─" * 96)
    for _e in _ETQ:
        _s = f"  {_e:<32}"
        for _c in _NIVS:
            _v = fmt.format(_p.loc[_e, _c])
            _s += f"{('▸' + _v) if _win[_c] == _e else _v:>13}"
        print(_s)
    print("\n   ▸ = mejor criterio de esa columna")
    return _p


_m_ski = _matriz('skill', 'MATRIZ 1 · SKILL vs «Norma·real» — ¿le gana a la norma con demanda perfecta?',
                 'SS = 1 − WAPE_criterio / WAPE_vara.  0 = empata con la vara.  '
                 '>0 = LE GANA.', '{:+.0%}')
_m_err = _matriz('err_p50', 'MATRIZ 2 · ERROR TÍPICO POR CELDA — mediana del |desvío %|',
                 'Esto SÍ varía con el nivel: mide celda a celda, no el total. '
                 'Menor es mejor.', '{:.1%}', 'cero')
_m_exa = _matriz('exacta', 'MATRIZ 3 · % DE CELDAS CON EL NÚMERO EXACTO',
                 'Manda ABAJO, donde la celda tiene 1 ó 2 secciones y el % engaña.',
                 '{:.1%}')

# ── Lectura ─────────────────────────────────────────────────────────────────
_op, _g_ = '⭐ Norma·P50  (el operativo)', '⭐ grano'
_ses = {e: _n[c].sum() / _TOT_REAL - 1 for c, e in _CRIT}
_vara_ses = _n[_VARA].sum() / _TOT_REAL - 1
_ctxg = contexto_niveles[contexto_niveles['nivel'] == _g_].iloc[0]
_mej_ski = _m_ski[_g_].idxmax()
_mej_exa = _m_exa[_g_].idxmax()
_ganan = int((_m_ski > 0).sum().sum())
print(f"""
{'═' * 100}
CÓMO LEER ESTAS MATRICES
{'═' * 100}
  ① EL COSTO DEL MODELO DE DEMANDA — la única lectura limpia de la MATRIZ 1.
     «Norma·P50» es la vara con el P50 en vez de los inscritos reales: la norma
     se cancela y queda sólo el error de proyección.
        skill al grano del modelo : {_m_ski.loc[_op, _g_]:+.0%}
        exacta {_ctxg['exacta_vara']:.1%} (vara)  ->  {_m_exa.loc[_op, _g_]:.1%} (con el P50)
     Eso es lo que cuesta no conocer la demanda de antemano. Todo lo demás que
     falta hasta el 100% ya estaba en la vara: es del ESTÁNDAR.

     ⚠️ ARRIBA ESE MISMO SKILL SALE POSITIVO ({_m_ski.loc[_op, 'Período']:+.0%} al nivel Período), y NO
        significa que proyectar salga mejor que conocer la demanda de verdad.
        El modelo SOBRE-proyecta inscritos, y ese sesgo positivo cancela en parte
        el sub-dimensionamiento de la norma ({_vara_ses:+.1%} -> {_ses[_op]:+.1%} de desvío total).
        Son dos errores que se compensan al sumar, no uno que se corrige: si se
        arregla el estándar, el sesgo del modelo queda al descubierto. Al grano
        fino, donde no hay con qué cancelar, el skill vuelve a {_m_ski.loc[_op, _g_]:+.0%}.

  ② EL EFECTO DE RECALIBRAR EL DENOMINADOR — el resto de las filas. Ahí el skill
     mezcla dos cambios (demanda proyectada + otro denominador), así que no
     aísla nada, pero responde la pregunta práctica: ¿qué criterio se acerca más?
        · mejor skill al grano  : {_mej_ski}  ({_m_ski.loc[_mej_ski, _g_]:+.0%})
        · más celdas exactas    : {_mej_exa}  ({_m_exa.loc[_mej_exa, _g_]:.1%})

  ③ EL DESVÍO QUE NO SE ARREGLA — la vara ya abre {_vara_ses:+.1%} usando los inscritos
     VERDADEROS. Ningún cambio en el modelo de demanda toca ese número: sale de
     que las secciones reales no se abren al tamaño que fija la norma.

{'─' * 100}
  {'⚠️' if _ganan == 0 else '⭐'} {_ganan} de {_m_ski.size} combinaciones criterio×nivel le ganan a la vara.
""" + ("""     Ninguna recalibración del denominador mejora a la norma con demanda
     perfecta: el problema no está en qué número se divide, sino en que la
     decisión de abrir una sección no sigue una regla de tamaño."""
       if _ganan == 0 else
       """     Donde el skill es positivo, ese denominador describe mejor la práctica
     que el estándar normativo — es un candidato concreto a revisar la norma."""))

print("\n  → `secciones_por_nivel` (nivel × criterio, formato largo)")
print("    `contexto_niveles` (celdas, N típico y comportamiento de la vara)")


════════════════════════════════════════════════════════════════════════════════════════════════════
SECCIONES · CADA CRITERIO EN CADA NIVEL DE AGREGACIÓN
════════════════════════════════════════════════════════════════════════════════════════════════════
  55,983 filas · 23 períodos cerrados · 168,974 secciones reales
  vara = N_SECCIONES_TEORICAS · la misma norma con los inscritos REALES

────────────────────────────────────────────────────────────────────────────────────────────────────
TABLA A · DESVÍO TOTAL vs SECCIONES REALES  —  la comparativa del gráfico
────────────────────────────────────────────────────────────────────────────────────────────────────
  Un valor por criterio: agrupar no lo cambia, porque las sumas son las mismas.
  Negativo = abre de MENOS que lo que efectivamente se abrió.

  criterio                                Σ secciones    vs real
  REAL — lo que se abrió                      168,974
  ⚖️ Norma·real  (norma + inscritos REALES)      123,335     -27.0% 

In [9]:
proy.head()

,PERIODO,periodo_fuente,MATERIA,CURSO,ASIGNATURA_REGLA,COMPONENTE,CAMPUS,ins_lag_1y,P10,P50,...,ESTANDAR_PROM_FUENTE,ESTANDAR_PROMEDIO,N_SECCIONES_REALES,N_SECCIONES_TEORICAS_PROYECTADAS,N_SECCIONES_PROMEDIO,N_SECCIONES_P25,N_SECCIONES_P50,N_SECCIONES_P75,N_SECCIONES_P90,N_SECCIONES_TEORICAS
0,202620,202520,CBB2,0002,CBB20002,ELG,ONL,9532.0,6717,7736,...,sin_observacion,NaN,NaN,118.0,NaN,6.0,5.0,4.0,3.0,NaN
1,202620,202520,FORI,0002,FORI0002,TEO,TPC,1986.0,1770,2093,...,sin_observacion,NaN,NaN,24.0,NaN,86.0,37.0,27.0,24.0,NaN
2,202620,202520,FORI,0002,FORI0002,TEO,LPS,916.0,819,984,...,periodo_serie,1.0,NaN,12.0,895.0,29.0,18.0,14.0,12.0,NaN
3,202620,202520,FORI,0002,FORI0002,TEO,BES,898.0,665,938,...,sin_observacion,NaN,NaN,11.0,NaN,85.0,29.0,14.0,12.0,NaN
4,202620,202520,FORI,0002,FORI0002,TEO,PAP,948.0,791,933,...,sin_observacion,NaN,NaN,11.0,NaN,50.0,25.0,16.0,12.0,NaN


In [10]:
proy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22142 entries, 0 to 22141
Data columns (total 36 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   PERIODO                           22142 non-null  int64  
 1   periodo_fuente                    22142 non-null  int64  
 2   MATERIA                           22142 non-null  object 
 3   CURSO                             22142 non-null  object 
 4   ASIGNATURA_REGLA                  22142 non-null  object 
 5   COMPONENTE                        22142 non-null  object 
 6   CAMPUS                            22142 non-null  object 
 7   ins_lag_1y                        22142 non-null  float64
 8   P10                               22142 non-null  int64  
 9   P50                               22142 non-null  int64  
 10  P90                               22142 non-null  int64  
 11  PROB_SE_DICTA                     22142 non-null  float64
 12  P50_

In [11]:
proy.describe()

,PERIODO,periodo_fuente,ins_lag_1y,P10,P50,P90,PROB_SE_DICTA,P50_ESPERADO,horizonte,wape_ref,...,ESTANDAR_TEORICO,ESTANDAR_PROMEDIO,N_SECCIONES_REALES,N_SECCIONES_TEORICAS_PROYECTADAS,N_SECCIONES_PROMEDIO,N_SECCIONES_P25,N_SECCIONES_P50,N_SECCIONES_P75,N_SECCIONES_P90,N_SECCIONES_TEORICAS
count,22142.000000,22142.000000,22142.000000,22142.000000,22142.000000,22142.000000,22142.000000,22142.000000,22142.000000,22142.000000,...,22142.000000,81.000000,0.0,22142.000000,81.000000,22142.000000,22142.000000,22142.000000,22142.000000,0.0
mean,202686.646870,202586.646870,51.425210,36.735074,50.866137,69.288637,0.787544,46.361349,1.303631,0.368695,...,50.989071,7.459136,NaN,2.323819,49.641975,6.527098,3.789043,2.358730,2.006594,NaN
std,43.310003,43.310003,119.244534,85.382197,106.778618,139.156766,0.285141,103.841927,0.459836,0.133751,...,24.496337,13.480883,NaN,3.839364,127.293883,14.175647,9.459695,4.371401,3.235129,NaN
min,202620.000000,202520.000000,0.000000,0.000000,0.000000,1.000000,0.006000,0.000000,1.000000,0.021400,...,3.000000,1.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
25%,202625.000000,202525.000000,9.000000,6.000000,9.000000,14.000000,0.689000,6.000000,1.000000,0.341900,...,30.000000,1.000000,NaN,1.000000,3.000000,1.000000,1.000000,1.000000,1.000000,NaN
50%,202710.000000,202610.000000,24.000000,16.000000,24.000000,35.000000,0.936000,21.000000,1.000000,0.350500,...,60.000000,2.000000,NaN,1.000000,9.000000,2.000000,1.000000,1.000000,1.000000,NaN
75%,202720.000000,202620.000000,60.000000,43.000000,62.000000,87.000000,0.985000,56.000000,2.000000,0.401400,...,80.000000,7.500000,NaN,2.000000,35.000000,6.000000,3.000000,2.000000,2.000000,NaN
max,202725.000000,202625.000000,9532.000000,6717.000000,7736.000000,10656.000000,0.999000,7572.000000,2.000000,2.333800,...,80.000000,65.000000,NaN,118.000000,895.000000,368.000000,193.000000,106.000000,106.000000,NaN


In [12]:
serie.head()

,PERIODO,MATERIA,CURSO,ASIGNATURA_REGLA,COMPONENTE,CAMPUS,INSCRITOS_REAL,PROY_BAJO,PROY_ESTANDAR,PROY_ALTO,...,ESTANDAR_PROM_FUENTE,ESTANDAR_PROMEDIO,N_SECCIONES_REALES,N_SECCIONES_TEORICAS_PROYECTADAS,N_SECCIONES_PROMEDIO,N_SECCIONES_P25,N_SECCIONES_P50,N_SECCIONES_P75,N_SECCIONES_P90,N_SECCIONES_TEORICAS
0,202205,ADPA,0001,ADPA0001,TEO,BES,19.0,5.0,15.0,35.0,...,periodo_serie,19.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0
1,202205,ADPA,0001,ADPA0001,TEO,PAP,30.0,5.0,13.0,28.0,...,periodo_serie,30.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0
2,202205,ADPA,0002,ADPA0002,TEO,BES,19.0,5.0,14.0,35.0,...,periodo_serie,19.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0
3,202205,ADPA,0003,ADPA0003,LAB,BES,19.0,6.0,15.0,35.0,...,periodo_serie,19.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0
4,202205,ADPA,0003,ADPA0003,LAB,PAP,30.0,4.0,14.0,32.0,...,periodo_serie,30.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0


In [13]:
serie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78125 entries, 0 to 78124
Data columns (total 35 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   PERIODO                           78125 non-null  int64  
 1   MATERIA                           78125 non-null  object 
 2   CURSO                             78125 non-null  object 
 3   ASIGNATURA_REGLA                  78125 non-null  object 
 4   COMPONENTE                        78125 non-null  object 
 5   CAMPUS                            78125 non-null  object 
 6   INSCRITOS_REAL                    55983 non-null  float64
 7   PROY_BAJO                         78125 non-null  float64
 8   PROY_ESTANDAR                     78125 non-null  float64
 9   PROY_ALTO                         78125 non-null  float64
 10  TIPO                              78125 non-null  object 
 11  horizonte                         78125 non-null  int64  
 12  wape

In [14]:
serie.describe()

,PERIODO,INSCRITOS_REAL,PROY_BAJO,PROY_ESTANDAR,PROY_ALTO,horizonte,wape_ref,ERROR_ABS,ERROR_PCT,DENTRO_BANDA,...,ESTANDAR_TEORICO,ESTANDAR_PROMEDIO,N_SECCIONES_REALES,N_SECCIONES_TEORICAS_PROYECTADAS,N_SECCIONES_PROMEDIO,N_SECCIONES_P25,N_SECCIONES_P50,N_SECCIONES_P75,N_SECCIONES_P90,N_SECCIONES_TEORICAS
count,78125.000000,55983.000000,78125.000000,78125.000000,78125.000000,78125.000000,22142.000000,55983.000000,55983.000000,55983.000000,...,78125.000000,56064.000000,55983.000000,78125.000000,56064.000000,78125.000000,78125.000000,78125.000000,78125.000000,55983.000000
mean,202476.904768,46.670775,30.318694,49.270170,73.464986,0.369472,0.368695,16.596031,0.775979,0.751532,...,51.582400,20.463373,3.018309,2.286182,3.687643,7.136358,3.770150,2.371930,1.975693,2.203080
std,172.649147,74.392981,59.219087,81.968859,112.669949,0.636456,0.133751,39.369790,3.679904,0.432129,...,23.960077,18.196777,6.756714,3.824763,8.961838,14.316819,8.782486,4.424313,3.129910,3.737732
min,202205.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.021400,0.000899,-1.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,202320.000000,10.000000,6.000000,12.000000,20.000000,0.000000,0.341900,2.679946,-0.139868,1.000000,...,30.000000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,202505.000000,23.000000,14.000000,26.000000,42.000000,0.000000,0.350500,6.633948,0.110337,1.000000,...,60.000000,15.000000,1.000000,1.000000,2.000000,3.000000,1.000000,1.000000,1.000000,1.000000
75%,202620.000000,56.000000,36.000000,61.000000,93.000000,1.000000,0.401400,16.085013,0.543719,1.000000,...,80.000000,28.000000,2.000000,2.000000,3.000000,7.000000,3.000000,2.000000,2.000000,2.000000
max,202725.000000,2523.000000,6717.000000,7736.000000,10656.000000,2.000000,2.333800,1564.564465,160.152362,1.000000,...,80.000000,788.000000,181.000000,118.000000,895.000000,502.000000,208.000000,119.000000,106.000000,73.000000


In [15]:
## ══════════════════════════════════════════════════════════════════════════════
## SECCIONES POR PERÍODO — real vs los tres criterios de estimación
## ══════════════════════════════════════════════════════════════════════════════
## Cuatro líneas sobre los 30 períodos de `serie`, con filtros de período, campus y
## componente. Responde: ¿cuántas secciones abriríamos con cada criterio, cuántas se
## abrieron de verdad, y cuánto de la diferencia es culpa del modelo?
##
## LAS TRES LÍNEAS BASE — cada una neutraliza una fuente de error distinta:
##
##   línea                          numerador   denominador    qué aísla
##   Reales                         —           —              la vara
##   Norma · demanda REAL           real        est. teórico   error del ESTÁNDAR
##   Norma · demanda P50 ⭐         P50         est. teórico   error TOTAL (la que programa)
##
## `N_SECCIONES_PROMEDIO` queda FUERA del gráfico (sigue en el dataset): su
## denominador es el estándar de la propia fila, así que contesta "¿cómo se llena
## ESTA asignatura?" mientras los escenarios contestan "¿cómo se llenan las de
## este tipo?". Mezclarlas en un mismo eje confundía dos preguntas distintas.
##
## ⭐ Sólo la tercera cubre los 7 períodos futuros. Las otras dos SE CORTAN:
##      · Reales             hasta 202715 (los h2 no tienen homólogo observado)
##      · Norma·demanda REAL hasta 202615 (necesita inscritos reales)
##    Se cortan en vez de caer a cero: un cero diría "no se abre ninguna
##    sección", que es falso.
##
## ⭐ FILTRO «Escenario»: agrega hasta 4 líneas más, con el denominador puesto en
##    el percentil de ocupación de ese PERIODO × COMPONENTE × CAMPUS (viene del
##    pipeline, celda 24c). Responden "¿cuántas secciones si las lleno como el 25%
##    menos ocupado, o como el 90% más lleno?". Arrancan apagadas.
##      p25 +144% vs lo real · p50 +25% · p75 −21% · p90 −35%
##    ⚠️ El grano es COMPONENTE, no la fila: al grano fino el 79% de las
##       combinaciones tiene <=2 secciones y los percentiles colapsan.
##
## ⚠️ Los errores NO son aditivos. El estándar teórico es grueso (80 alumnos) y
##    ABSORBE el error del modelo; con el estándar real (mediana 17) cada error
##    de demanda sí mueve secciones. Por eso la línea naranja se separa tanto:
##    no es que el modelo empeore, es que deja de estar enmascarado.
## ──────────────────────────────────────────────────────────────────────────────
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display

# `N_SECCIONES_PROMEDIO` SIGUE en el dataset pero NO se grafica: su denominador es
# el estándar de la propia fila, así que responde "¿cómo se llena ESTA asignatura?"
# — otra pregunta que las demás. Los escenarios por percentil ocupan su lugar y su
# color, y contestan "¿cómo se llenan las secciones de este tipo?".
_SER = ['N_SECCIONES_REALES', 'N_SECCIONES_TEORICAS',
        'N_SECCIONES_TEORICAS_PROYECTADAS']
# Etiquetas CORTAS a propósito: con textos largos la leyenda se parte en dos filas
# y se solapa con el título. Qué aísla cada línea se explica en el subtítulo HTML.
_ETQ = {'N_SECCIONES_REALES':               'Reales',
        'N_SECCIONES_TEORICAS':             'Norma · demanda real',
        'N_SECCIONES_TEORICAS_PROYECTADAS': '⭐ Norma · P50'}
# Las dos normativas comparten familia de color a propósito: son la MISMA fórmula
# con distinta demanda, y su separación es justo lo que hay que leer.
_COL = {'N_SECCIONES_REALES':               '#2f3e4d',   # el dato duro, sobrio
        'N_SECCIONES_TEORICAS':             '#14607d',   # azul oscuro: demanda real
        'N_SECCIONES_TEORICAS_PROYECTADAS': '#63aac6'}   # azul claro: demanda modelo
_DASH = {'N_SECCIONES_REALES': None, 'N_SECCIONES_TEORICAS': 'dash',
         'N_SECCIONES_TEORICAS_PROYECTADAS': None}
_ANCHO = {'N_SECCIONES_REALES': 2.6, 'N_SECCIONES_TEORICAS': 2.0,
          'N_SECCIONES_TEORICAS_PROYECTADAS': 2.8}

# ── Escenarios por percentil de ocupación (opcionales) ───────────────────────
# Denominador = tamaño típico de sección de ese PERIODO × COMPONENTE × CAMPUS,
# según el cuartil. Comparten gama de color a propósito: se leen como UNA banda,
# no como cuatro criterios distintos. P25 = secciones vacías -> muchas secciones.
_PCT = [25, 50, 75, 90]
_SER_PCT = [f'N_SECCIONES_P{p}' for p in _PCT]
# gama cálida, graduada de claro a oscuro: el escenario más vacío es el más
# tenue y el más lleno el más intenso, así el abanico se lee como una banda.
_COL_PCT = {25: '#e3ad7e', 50: '#c0703a', 75: '#93511f', 90: '#5f330f'}
_ETQ_PCT = {25: 'p25 · secciones vacías', 50: 'p50 · mediana',
            75: 'p75 · secciones llenas', 90: 'p90 · muy llenas'}

_PER = sorted(serie['PERIODO'].unique())
_CAM = sorted(serie['CAMPUS'].dropna().unique())
_CMP = sorted(serie['COMPONENTE'].dropna().unique())
_ULT_REAL = serie.loc[serie['TIPO'] == 'REAL', 'PERIODO'].max()


def _agrega(campus, comps, periodos):
    """Suma de secciones por período. Devuelve NaN donde el dato real no existe.

    `periodos` es una LISTA, no un rango: la selección puede ser discontinua
    (p. ej. solo los TT 10 de cada año) y el eje respeta ese orden.

    """
    d = serie[serie['CAMPUS'].isin(campus) & serie['COMPONENTE'].isin(comps) &
              serie['PERIODO'].isin(periodos)]
    if not len(d):
        return None, 0
    _cols = _SER + [c for c in _SER_PCT if c in d.columns]
    g = d.groupby('PERIODO', observed=True)[_cols].sum(min_count=1)
    # Si en un período casi ninguna fila trae el dato, no se dibuja: no es un cero.
    # Aplica a las dos columnas que dependen de observaciones que pueden no existir.
    for _c in ('N_SECCIONES_REALES', 'N_SECCIONES_TEORICAS'):
        _cob = d.groupby('PERIODO', observed=True)[_c].apply(lambda s: s.notna().mean())
        g.loc[_cob < 0.5, _c] = np.nan
    return g, len(d)


_fig = go.FigureWidget()
for _c in _SER:
    _fig.add_scatter(x=[], y=[], name=_ETQ[_c], mode='lines+markers',
                     line=dict(color=_COL[_c], width=_ANCHO[_c], dash=_DASH[_c]),
                     marker=dict(size=5),
                     hovertemplate='%{x}<br>%{y:,.0f} secciones<extra>%{fullData.name}</extra>')
for _p in _PCT:                       # arrancan ocultas: se activan con el filtro
    _fig.add_scatter(x=[], y=[], name=_ETQ_PCT[_p], mode='lines', visible=False,
                     line=dict(color=_COL_PCT[_p], width=1.6, dash='dashdot'),
                     hovertemplate='%{x}<br>%{y:,.0f} secciones<extra>%{fullData.name}</extra>')
_fig.update_layout(
    template='plotly_white', height=500,
    # t=92 deja sitio para dos franjas: el título arriba y la leyenda debajo.
    # Con menos margen la leyenda sube y pisa el título.
    margin=dict(l=64, r=30, t=92, b=56),
    title=dict(x=0, xanchor='left', y=0.97, yanchor='top',
               font=dict(size=13.5, color='#41505f')),
    xaxis=dict(title='Período', type='category', tickangle=-45),
    yaxis=dict(title='Secciones', rangemode='tozero', separatethousands=True),
    legend=dict(orientation='h', yanchor='bottom', y=1.015, x=0,
                font=dict(size=11.5), itemwidth=30),
    hovermode='x unified')

# ── Controles ────────────────────────────────────────────────────────────────
# Los tres filtros son SelectMultiple: misma mecánica, ctrl/shift para elegir varios.
# El de período permite selecciones DISCONTINUAS — p. ej. solo los TT 10 de cada año.
_w_per = W.SelectMultiple(options=[(str(p), p) for p in _PER], value=tuple(_PER),
                          description='Período', rows=10,
                          layout=W.Layout(width='190px'))
_w_cam = W.SelectMultiple(options=_CAM, value=tuple(_CAM), description='Campus',
                          rows=10, layout=W.Layout(width='190px'))
_w_cmp = W.SelectMultiple(options=_CMP, value=tuple(_CMP), description='Compon.',
                          rows=10, layout=W.Layout(width='195px'))
_w_all = W.Button(description='Todo', layout=W.Layout(width='90px'))
# Escenarios de ocupación: por defecto NINGUNO, para no saturar el gráfico.
_w_pct = W.SelectMultiple(options=[(_ETQ_PCT[p], p) for p in _PCT], value=(),
                          description='Escenario', rows=4,
                          layout=W.Layout(width='260px'))
# atajos para los cortes que más se usan
_w_reg = W.Button(description='TT 10/20', layout=W.Layout(width='90px'),
                  tooltip='Solo los períodos de pregrado regular')
_w_adv = W.Button(description='TT 05/15/25', layout=W.Layout(width='110px'),
                  tooltip='Solo los períodos advance')
_w_msg = W.HTML()


def _redibuja(*_):
    _sel = sorted(_w_per.value)
    g, n = _agrega(list(_w_cam.value), list(_w_cmp.value), _sel)
    if g is None or not len(g):
        _w_msg.value = "<b style='color:#a34b3c'>Sin datos para ese filtro</b>"
        with _fig.batch_update():
            for t in _fig.data:
                t.x, t.y = [], []
        return
    _x = [str(i) for i in g.index]
    with _fig.batch_update():
        for t, c in zip(_fig.data, _SER):
            t.x, t.y = _x, g[c].values
        # las de percentil sólo se dibujan si están elegidas en el filtro
        for _t, _p in zip(_fig.data[len(_SER):], _PCT):
            if _p in _w_pct.value and f'N_SECCIONES_P{_p}' in g.columns:
                _t.x, _t.y, _t.visible = _x, g[f'N_SECCIONES_P{_p}'].values, True
            else:
                _t.x, _t.y, _t.visible = [], [], False
        # Marca dónde termina lo observado y empieza lo proyectado.
        # ⚠️ Se construye el shape a mano en vez de `add_vline`: sobre un
        #    FigureWidget dentro de `batch_update`, add_vline falla al resolver el
        #    yref después de vaciar `layout.shapes` (TypeError sobre None += str).
        # con selección discontinua la divisoria sigue valiendo: separa los
        # períodos observados de los proyectados dentro de lo que se está viendo
        _sh, _an = [], []
        _n_real = sum(1 for p in g.index if p <= _ULT_REAL)
        if 0 < _n_real < len(g):
            _i = _n_real - 0.5
            _sh = [dict(type='line', xref='x', yref='paper', x0=_i, x1=_i,
                        y0=0, y1=1, line=dict(color='#9aa8b4', width=1, dash='dot'))]
            _an = [dict(x=_i, xref='x', yref='paper', y=1.0, text=' proyectado →',
                        showarrow=False, xanchor='left',
                        font=dict(size=11, color='#6d7c8b'))]
        _fig.layout.shapes = _sh
        _fig.layout.annotations = _an
        # el nombre del gráfico vive en el H3 de arriba; aquí solo los filtros
        _fig.layout.title.text = (
            f"{len(_sel)}/{len(_PER)} períodos · {len(_w_cam.value)}/{len(_CAM)} sedes"
            f" · {len(_w_cmp.value)}/{len(_CMP)} componentes")
    _r  = g['N_SECCIONES_REALES'].sum()
    _tp = g['N_SECCIONES_TEORICAS_PROYECTADAS'].sum()
    if not _r:
        _w_msg.value = (f"<span style='font-family:monospace'>{n:,} filas · "
                        f"sin secciones reales en el rango</span>")
        return
    _s = (f"<span style='font-family:monospace;font-size:12.5px'>{n:,} filas · "
          f"Σ reales <b>{_r:,.0f}</b> · la que programa <b>{_tp:,.0f}</b> "
          f"({_tp/_r-1:+.1%})")
    # La descomposición sólo vale donde las tres existen a la vez: los períodos
    # cerrados. Compararlas sobre rangos distintos daría una brecha inventada.
    _c3 = ['N_SECCIONES_REALES', 'N_SECCIONES_TEORICAS',
           'N_SECCIONES_TEORICAS_PROYECTADAS']
    _amb = g[_c3].dropna()
    if len(_amb):
        _rr = _amb['N_SECCIONES_REALES'].sum()
        _s += (f"<br><span style='color:#14607d'>&nbsp;&nbsp;sobre los {len(_amb)} "
               f"períodos cerrados · error del ESTÁNDAR "
               f"<b>{_amb['N_SECCIONES_TEORICAS'].sum()/_rr-1:+.1%}</b>"
               f" · TOTAL <b>{_amb['N_SECCIONES_TEORICAS_PROYECTADAS'].sum()/_rr-1:+.1%}</b>"
               f"</span>")
    if _w_pct.value:
        _ab = [f"{_ETQ_PCT[_p].split(' · ')[0]} <b>{g[f'N_SECCIONES_P{_p}'].sum():,.0f}</b>"
               for _p in sorted(_w_pct.value) if f'N_SECCIONES_P{_p}' in g.columns]
        _s += ("</span><br><span style='color:#6f4e91'>&nbsp;&nbsp;escenarios · "
               + " · ".join(_ab)
               + f" &nbsp;(la norma: {_tp:,.0f})")
    _w_msg.value = _s + "</span>"
    if _w_pct.value:
        _ab = [f"{_ETQ_PCT[_p].split(' · ')[0]} <b>{g[f'N_SECCIONES_P{_p}'].sum():,.0f}</b>"
               for _p in sorted(_w_pct.value) if f'N_SECCIONES_P{_p}' in g.columns]
        _s += ("</span><br><span style='color:#6f4e91'>&nbsp;&nbsp;escenarios · "
               + " · ".join(_ab)
               + f" &nbsp;(la norma: {_tp:,.0f})")
    _w_msg.value = _s + "</span>"


def _reset(_):
    _w_cam.value, _w_cmp.value = tuple(_CAM), tuple(_CMP)
    _w_per.value = tuple(_PER)


def _solo_tt(tts):
    """Deja seleccionados solo los períodos de esos tipos de término."""
    return lambda _: setattr(_w_per, 'value',
                             tuple(p for p in _PER if p % 100 in tts))


for _w in (_w_per, _w_cam, _w_cmp, _w_pct):
    _w.observe(_redibuja, names='value')
_w_all.on_click(_reset)
_w_reg.on_click(_solo_tt({10, 20}))
_w_adv.on_click(_solo_tt({5, 15, 25}))
_redibuja()

display(W.VBox([
    W.HTML("<h3 style='margin:2px 0 2px'>Secciones por período</h3>"
           "<div style='color:#6d7c8b;font-size:12.5px;margin-bottom:6px'>"
           "La <b>sólida gris</b> es el dato. La <b style='color:#63aac6'>azul clara "
           "sólida</b> es la que se usa para programar. Las otras dos aíslan cada "
           "fuente de error: la <b style='color:#14607d'>azul oscura</b> usa la "
           "demanda real (queda el error del <b>estándar</b>) y la "
           "<b style='color:#c0703a'>naranja</b> usa el estándar real (queda el "
           "error del <b>modelo</b>). Tres se cortan donde no hay observación."
           "</div>"),
    W.HBox([_w_per, _w_cam, _w_cmp,
            W.VBox([_w_all, _w_reg, _w_adv],
                   layout=W.Layout(padding='18px 0 0 10px')),
            W.VBox([_w_pct], layout=W.Layout(padding='18px 0 0 14px'))]),
    _w_msg, _fig]))


In [16]:
## ══════════════════════════════════════════════════════════════════════════════
## TAMAÑO DE SECCIÓN OBSERVADO vs ESTÁNDAR NORMATIVO
## ══════════════════════════════════════════════════════════════════════════════
## DE DÓNDE SALE CADA NÚMERO — la cadena completa, en cuatro pasos:
##
##   ① PIPELINE, celda 24c · sobre el DETALLE de inscripción (una fila = un alumno
##     en un NRC), cuenta los alumnos de cada sección:
##         grano  PERIODO × NRC × MATERIA × CURSO × COMPONENTE × CAMPUS
##         ⚠️ PERIODO entra en la llave porque el NRC se REUTILIZA entre períodos
##            (el 77% reaparece): sin él se colapsarían secciones distintas.
##
##   ② Sobre esa distribución de tamaños saca los cuartiles:
##         grano  PERIODO × COMPONENTE × CAMPUS   ->  estandar_p25/50/75/90
##         Es decir: "en este período, en teoría, en TPC, ¿de qué tamaño fueron
##         las secciones?". Son 2.392 grupos, un valor por grupo.
##         ⚠️ NO es un rango de años: cada período tiene los suyos. El p50 de TEO
##            en TPC va de 13 (advance) a 55 (regular) según el TT.
##
##   ③ ESTA CELDA agrega esos valores a nivel COMPONENTE para poder mirarlos:
##         · la columna `p25..p90` es la MEDIANA de los p25..p90 de cada período
##           -> es un resumen de resúmenes, no el percentil de todas las secciones
##         · `ocup` sí es directo: Σ inscritos / Σ secciones del componente
##
##   ④ El ESTÁNDAR NORMATIVO no se calcula: lo fija la Dirección (CONTEXTO §8.2).
##         TEO se desdobla — 80 en TT 10/20 (pregrado regular), 60 en 05/15/25.
##         La columna `norma` muestra la mediana de lo que aplica a cada fila.
##
## LA PREGUNTA QUE RESPONDE: ¿la norma describe una práctica que existe?
##   Si cae dentro de p25–p90, sí. Si queda sobre el p90, está pidiendo secciones
##   más llenas de las que nunca se llenan, y la conversión abrirá de menos
##   sistemáticamente — no por error del modelo, sino por la norma misma.
##
## VENTANA: últimos 3 años cerrados, sólo TT regulares (05/10/15/20/25).
## ──────────────────────────────────────────────────────────────────────────────
_PCOLS = [f'estandar_p{p}' for p in [25, 50, 75, 90]]
_sp = pd.read_parquet(f'{CARPETA}/DEMANDA_PROCESADA.parquet',
                      columns=['PERIODO', 'COMPONENTE', 'CAMPUS', 'INSCRITOS',
                               'n_secciones'] + _PCOLS)
_A_MAX = _sp['PERIODO'].max() // 100
_sp = _sp[(_sp['PERIODO'] % 100).isin([5, 10, 15, 20, 25]) &
          (_sp['PERIODO'] // 100 >= _A_MAX - 2)]
_et = proy.groupby('COMPONENTE', observed=True)['ESTANDAR_TEORICO'].median()

print("═" * 88)
print("TAMAÑO DE SECCIÓN OBSERVADO vs ESTÁNDAR NORMATIVO")
print("═" * 88)
print(f"  ventana: {_sp['PERIODO'].min()}–{_sp['PERIODO'].max()} · "
      f"{_sp['PERIODO'].nunique()} períodos · {len(_sp):,} filas del panel")
print("  p25..p90 = MEDIANA, entre períodos, del percentil de cada período")
print("             (el percentil se calcula sobre alumnos por NRC, en el pipeline)")
print("  ocup     = Σ inscritos / Σ secciones — directo, sin percentilar")
print("  norma    = estándar fijado por la Dirección (no se calcula)\n")

_g = (_sp.groupby('COMPONENTE', observed=True)
         .agg(secciones=('n_secciones', 'sum'), inscritos=('INSCRITOS', 'sum'),
              periodos=('PERIODO', 'nunique'),
              **{f'p{p}': (f'estandar_p{p}', 'median') for p in [25, 50, 75, 90]},
              # cuánto se mueve el p50 ENTRE períodos: si es mucho, la mediana
              # de arriba está resumiendo cosas muy distintas
              p50_min=('estandar_p50', 'min'), p50_max=('estandar_p50', 'max'))
         .join(_et.rename('norma')))
_g['ocup'] = _g['inscritos'] / _g['secciones']
_g['abanico'] = _g['p90'] / _g['p25'].clip(lower=1)
_g = _g[_g['secciones'] > 500].sort_values('secciones', ascending=False)


def _sitúa(r):
    """Dónde cae la norma dentro de la distribución observada."""
    if pd.isna(r['norma']):   return '—'
    if r['norma'] > r['p90']: return '⚠️ SOBRE el p90'
    if r['norma'] > r['p75']: return 'entre p75 y p90'
    if r['norma'] > r['p50']: return 'entre p50 y p75'
    if r['norma'] > r['p25']: return 'entre p25 y p50'
    return '⚠️ BAJO el p25'


_g['dónde cae'] = _g.apply(_sitúa, axis=1)

print(f"  {'comp':<6} {'secc':>7} {'norma':>6} | {'p25':>5} {'p50':>5} {'p75':>5} {'p90':>5} "
      f"| {'p50 entre per.':>15} | {'ocup':>6} {'aban':>6}  {'la norma cae'}")
print("  " + "─" * 86)
for _c, _r in _g.iterrows():
    _mark = '  ⚠️' if _r['norma'] > _r['p90'] else ''
    _rango = '{:.0f}–{:.0f}'.format(_r['p50_min'], _r['p50_max'])
    print(f"  {_c:<6} {_r['secciones']:>7,.0f} {_r['norma']:>6.0f} | "
          f"{_r['p25']:>5.0f} {_r['p50']:>5.0f} {_r['p75']:>5.0f} {_r['p90']:>5.0f} | "
          f"{_rango:>15} | "
          f"{_r['ocup']:>6.1f} {_r['abanico']:>5.1f}x  {_r['dónde cae']}{_mark}")

_mal = _g[_g['norma'] > _g['p90']]
print(f"""
  ── CÓMO SE LEE ──────────────────────────────────────────────────────────────
   · norma DENTRO de p25–p90  ->  describe una práctica que existe
   · norma SOBRE el p90       ->  ⚠️ ni el 10% de secciones más llenas la alcanza
   · abanico ESTRECHO (~1x)   ->  hay tope efectivo: las secciones son homogéneas
   · «p50 entre per.» ancho   ->  la mediana de arriba resume períodos muy
                                  distintos; suele ser el corte regular/advance

  {len(_mal)} de {len(_g)} componentes tienen la norma sobre su p90"""
      + (":\n     " + '\n     '.join(
          f"{c}: norma {r['norma']:.0f} vs p90 {r['p90']:.0f} "
          f"({r['norma']/r['p90']-1:+.0%})" for c, r in _mal.iterrows())
         if len(_mal) else ''))

# ── El mismo corte por sede, donde la decisión de programar se toma ──────────
print("\n" + "═" * 88)
print("POR SEDE — los tres componentes de mayor volumen")
print("═" * 88)
print("  Mismo cálculo, abriendo por CAMPUS: es el grano al que existe el percentil.\n")
_top = _g.head(3).index.tolist()
_gs = (_sp[_sp['COMPONENTE'].isin(_top)]
       .groupby(['COMPONENTE', 'CAMPUS'], observed=True)
       .agg(secc=('n_secciones', 'sum'),
            **{f'p{p}': (f'estandar_p{p}', 'median') for p in [25, 50, 90]}))
_gs = _gs[_gs['secc'] > 200]
for _c in _top:
    if _c not in _gs.index.get_level_values(0):
        continue
    _n = _et.get(_c, np.nan)
    _sub = _gs.loc[[_c]].sort_values('secc', ascending=False)
    _n_mal = (_n > _sub['p90']).sum()
    print(f"  {_c}  ·  norma {_n:.0f}  ·  la supera el p90 en {_n_mal} de {len(_sub)} sedes")
    print(f"    {'sede':<6} {'secciones':>10} {'p25':>6} {'p50':>6} {'p90':>6}")
    for (_, _cam), _r in _sub.iterrows():
        _al = '   ⚠️ norma > p90' if _n > _r['p90'] else ''
        print(f"    {_cam:<6} {_r['secc']:>10,.0f} {_r['p25']:>6.0f} "
              f"{_r['p50']:>6.0f} {_r['p90']:>6.0f}{_al}")
    print()

tabla_percentiles = _g.copy()
print("  → disponible como `tabla_percentiles`")


════════════════════════════════════════════════════════════════════════════════════════
TAMAÑO DE SECCIÓN OBSERVADO vs ESTÁNDAR NORMATIVO
════════════════════════════════════════════════════════════════════════════════════════
  ventana: 202405–202620 · 14 períodos · 41,219 filas del panel
  p25..p90 = MEDIANA, entre períodos, del percentil de cada período
             (el percentil se calcula sobre alumnos por NRC, en el pipeline)
  ocup     = Σ inscritos / Σ secciones — directo, sin percentilar
  norma    = estándar fijado por la Dirección (no se calcula)

  comp      secc  norma |   p25   p50   p75   p90 |  p50 entre per. |   ocup   aban  la norma cae
  ──────────────────────────────────────────────────────────────────────────────────────
  LAB     32,405     30 |     7    15    26    30 |            1–25 |   16.1   4.3x  entre p75 y p90
  TEO     26,064     80 |    13    30    54    71 |            1–61 |   34.1   5.5x  ⚠️ SOBRE el p90  ⚠️
  OCS     23,056      3 |     1     1    

In [17]:
## ══════════════════════════════════════════════════════════════════════════════
## EL MISMO GRÁFICO, EN TABLA — período a período, con los mismos filtros
## ══════════════════════════════════════════════════════════════════════════════
## Reutiliza `_agrega()` de la celda del gráfico, así que las cifras son las mismas
## por construcción: si la tabla y el gráfico difirieran, sería un bug.
##
## Para qué tener las dos vistas: el gráfico muestra la FORMA (tendencia,
## estacionalidad, dónde se abre el abanico); la tabla da el NÚMERO exacto, que es
## lo que se copia a un correo o a una minuta.
##
## ⚠️ El HTML se arma con `.format()`, NO con f-strings: mezclar llaves de CSS con
##    llaves de interpolación es frágil — un reemplazo de texto sobre un f-string
##    partido dejó `{_txt}` literal en la tabla (2026-08-31).
## ⚠️ Colores en rgba y `opacity`, nunca hex opaco: el notebook puede estar en tema
##    oscuro y un fondo claro fijo dejaría la tabla ilegible.
## ──────────────────────────────────────────────────────────────────────────────
_BRD, _BRD2 = "1px solid rgba(128,128,128,.35)", "2px solid rgba(128,128,128,.7)"
_BG_FUT = "rgba(192,112,58,.10)"          # marca tenue de "período proyectado"
_VERDE, _AMBAR, _ROJO = '#5ab07d', '#d9a441', '#e06c5a'

_T_BASE = [('N_SECCIONES_REALES', 'Reales'),
           ('N_SECCIONES_TEORICAS', 'Norma·real'),
           ('N_SECCIONES_TEORICAS_PROYECTADAS', '⭐ Norma·P50')]

_t_per = W.SelectMultiple(options=[(str(p), p) for p in _PER], value=tuple(_PER),
                          description='Período', rows=10, layout=W.Layout(width='190px'))
_t_cam = W.SelectMultiple(options=_CAM, value=tuple(_CAM), description='Campus',
                          rows=10, layout=W.Layout(width='190px'))
_t_cmp = W.SelectMultiple(options=_CMP, value=tuple(_CMP), description='Compon.',
                          rows=10, layout=W.Layout(width='195px'))
_t_pct = W.SelectMultiple(options=[(_ETQ_PCT[p], p) for p in _PCT], value=(),
                          description='Escenario', rows=4, layout=W.Layout(width='260px'))
_t_all = W.Button(description='Todo', layout=W.Layout(width='90px'))
# mismos atajos que el gráfico: los dos regímenes se miran por separado casi siempre
_t_reg = W.Button(description='TT 10/20', layout=W.Layout(width='90px'),
                  tooltip='Solo los períodos de pregrado regular')
_t_adv = W.Button(description='TT 05/15/25', layout=W.Layout(width='110px'),
                  tooltip='Solo los períodos advance')
_t_out = W.HTML()

_CELDA = "<{tag} style='text-align:{a};padding:{p};border-{lado}:{b};{e}'>{t}</{tag}>"


def _vs(val, real):
    """El '% vs real' que va bajo cada cifra. Devuelve '' si no es comparable.

    ⚠️ En los períodos proyectados `N_SECCIONES_REALES` es la referencia del
       HOMÓLOGO, no el dato de ese período: el % ahí compara contra el año
       anterior, no contra la realidad. Por eso esas filas van sombreadas.
    """
    if pd.isna(val) or pd.isna(real) or not real:
        return ''
    _d = val / real - 1
    _cl = _ROJO if abs(_d) > .25 else (_AMBAR if abs(_d) > .1 else _VERDE)
    return ("<div style='font-size:10.5px;color:{};margin-top:1px'>{:+.1%}</div>"
            .format(_cl, _d))


def _c(txt, a='right', tag='td', b=None, lado='bottom', p='4px 10px', e=''):
    return _CELDA.format(tag=tag, a=a, p=p, lado=lado, b=b or _BRD, e=e, t=txt)


def _tabla(*_):
    g, n = _agrega(list(_t_cam.value), list(_t_cmp.value), sorted(_t_per.value))
    if g is None or not len(g):
        _t_out.value = "<b style='color:{}'>Sin datos para ese filtro</b>".format(_ROJO)
        return
    _cols = list(_T_BASE) + [('N_SECCIONES_P{}'.format(p), _ETQ_PCT[p].split(' · ')[0])
                             for p in sorted(_t_pct.value)]
    h = ["<table style='border-collapse:collapse;font-family:ui-monospace,monospace;"
         "font-size:12.5px'><thead><tr>",
         _c('Período', 'left', 'th', _BRD2, p='5px 10px')]
    for _col, _etq in _cols:
        h.append(_c(_etq, 'right', 'th', _BRD2, p='5px 10px'))
    h.append("</tr></thead><tbody>")

    for _p, _r in g.iterrows():
        _fut = _p > _ULT_REAL
        h.append("<tr style='background:{}'>".format(_BG_FUT if _fut else 'transparent'))
        h.append(_c('{}{}'.format(_p, ' ·proy' if _fut else ''), 'left'))
        _rr = _r.get('N_SECCIONES_REALES')
        for _col, _ in _cols:
            _v = _r.get(_col, np.nan)
            _num = '—' if pd.isna(_v) else '{:,.0f}'.format(_v)
            # la columna de reales es la vara: no se compara consigo misma
            _pct = '' if _col == 'N_SECCIONES_REALES' else _vs(_v, _rr)
            h.append(_c(_num + _pct))
        h.append("</tr>")

    # fila de totales
    h.append("<tr>" + _c('Σ', 'left', 'td', _BRD2, 'top', '6px 10px', 'font-weight:600'))
    _tr = g['N_SECCIONES_REALES'].sum()
    for _col, _ in _cols:
        _s = g[_col].sum() if _col in g.columns else np.nan
        _pt = '' if _col == 'N_SECCIONES_REALES' else _vs(_s, _tr)
        h.append(_c(('' if pd.isna(_s) else '{:,.0f}'.format(_s)) + _pt,
                    'right', 'td', _BRD2, 'top', '6px 10px', 'font-weight:600'))
    h.append("</tr></tbody></table>")
    h.append("<div style='font-family:ui-monospace;font-size:11.5px;opacity:.65;"
             "margin-top:6px'>{:,} filas · {} períodos · las sombreadas son "
             "proyectadas · «—» = sin observación, no cero</div>".format(n, len(g)))
    _t_out.value = ''.join(h)


def _t_reset(_):
    _t_cam.value, _t_cmp.value = tuple(_CAM), tuple(_CMP)
    _t_per.value = tuple(_PER)


def _t_solo_tt(tts):
    """Deja seleccionados solo los períodos de esos tipos de término."""
    return lambda _: setattr(_t_per, 'value',
                             tuple(p for p in _PER if p % 100 in tts))


for _w in (_t_per, _t_cam, _t_cmp, _t_pct):
    _w.observe(_tabla, names='value')
_t_all.on_click(_t_reset)
_t_reg.on_click(_t_solo_tt({10, 20}))
_t_adv.on_click(_t_solo_tt({5, 15, 25}))
_tabla()

display(W.VBox([
    W.HTML("<h3 style='margin:2px 0 2px'>Secciones por período · tabla</h3>"
           "<div style='opacity:.7;font-size:12.5px;margin-bottom:6px'>"
           "Las mismas cifras del gráfico. <b>Bajo cada una, su desvío contra las "
           "reales</b>: <span style='color:{v}'>verde</span> bajo 10%, "
           "<span style='color:{a}'>ámbar</span> hasta 25%, "
           "<span style='color:{r}'>rojo</span> por encima. En las filas "
           "sombreadas la referencia es el <b>homólogo del año anterior</b>, no el "
           "período mismo.</div>"
           .format(v=_VERDE, a=_AMBAR, r=_ROJO)),
    W.HBox([_t_per, _t_cam, _t_cmp,
            W.VBox([_t_all, _t_reg, _t_adv],
                   layout=W.Layout(padding='18px 0 0 10px')),
            W.VBox([_t_pct], layout=W.Layout(padding='18px 0 0 14px'))]),
    _t_out]))
